In [ ]:
!git clone -b phase25b-maira2-feasibility \
    https://github.com/gvasu-2023/Evidence-State-Radiology.git \
    /content/Evidence-State-Radiology

Cloning into '/content/Evidence-State-Radiology'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (394/394), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 394 (delta 142), reused 348 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (394/394), 1.25 MiB | 11.29 MiB/s, done.
Resolving deltas: 100% (142/142), done.


In [ ]:
%cd /content/Evidence-State-Radiology

!git log --oneline -5

/content/Evidence-State-Radiology
5907f33 (HEAD -> phase25b-maira2-feasibility, origin/phase25b-maira2-feasibility) feat: add phases 17-25 reliability evaluation and MAIRA-2 feasibility
c4552c2 (origin/main, origin/HEAD) feat: freeze validated six-state benchmark
c094e8e feat: generate phase16c evidence-state benchmark
1f9c55a feat: select phase16b IU-Xray benchmark
fd65a6d feat: audit expanded IU-Xray benchmark


In [ ]:
from pathlib import Path

ROOT = Path("/content/Evidence-State-Radiology")

print("Repository:", ROOT.exists())

for directory in [
    "src/generation",
    "src/evaluation",
    "results/tables",
    "tests",
]:
    p = ROOT / directory
    print(f"{directory}: {p.exists()}")

Repository: True
src/generation: True
src/evaluation: True
results/tables: True
tests: True


In [ ]:
print("\nGeneration source files:")
for p in sorted((ROOT / "src" / "generation").glob("*.py")):
    print(" ", p.name)

print("\nEvaluation source files:")
for p in sorted((ROOT / "src" / "evaluation").glob("*.py")):
    print(" ", p.name)


Generation source files:
  __init__.py
  baseline.py
  run_baseline.py
  run_baseline_experiment.py
  run_iu_baseline.py
  test_context_conditioning.py

Evaluation source files:
  __init__.py
  analyze_conditions.py
  analyze_evidence_states.py
  audit_baseline.py
  audit_context_completeness.py
  audit_phase11_dataset.py
  build_reference_candidates.py
  claim_evaluator.py
  claim_extractor.py
  claim_patterns.py
  evaluate_abstention_metrics.py
  evaluate_analyzer.py
  evaluate_analyzer_phase12b.py
  evaluate_analyzer_phase12c.py
  evaluate_chexpert_f1.py
  evaluate_gated_generation.py
  evaluate_phase11_paired.py
  evaluate_phase13a_gate.py
  evaluate_phase13b_gated_reports.py
  evaluate_radgraph_f1.py
  evaluate_reference_factuality.py
  evaluate_reliability_gate.py
  evaluate_risk_coverage.py
  evaluate_unsupported_claims.py
  metrics.py
  run_phase15_evaluation.py
  run_phase19_claim_evaluation.py
  run_phase20_metrics.py
  run_phase21_risk_coverage.py
  run_phase22_statistics.p

In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

ROOT = Path("/content/Evidence-State-Radiology")

# Check the frozen Phase 16 benchmark
BENCHMARK = ROOT / "results" / "tables" / "phase16_perturbations.csv"

print("Benchmark:", BENCHMARK)
print("Exists:", BENCHMARK.exists())

if BENCHMARK.exists():
    df = pd.read_csv(BENCHMARK)

    print("\nShape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())

    print("\nConditions:")
    print(df["condition"].value_counts().sort_index())

    print("\nUnique studies:", df["uid"].nunique())

    print("\nFrontal images:", df["frontal_image"].nunique())

    print("\nBenchmark SHA-256:")

    h = hashlib.sha256()
    with open(BENCHMARK, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    print(h.hexdigest())

Benchmark: /content/Evidence-State-Radiology/results/tables/phase16_perturbations.csv
Exists: True

Shape: (600, 20)
Columns:
['sample_id', 'uid', 'condition', 'evidence_state', 'original_context', 'perturbed_context', 'frontal_image', 'lateral_image', 'findings', 'impression', 'comparison', 'source_context_uid', 'source_context_sample_id', 'removed_evidence_text', 'evidentiary_incomplete_method', 'conflict_target', 'conflict_method', 'conflict_source_text', 'generation_status', 'generation_failure_reason']

Conditions:
condition
conflicting               100
evidentiary_incomplete    100
insufficient              100
irrelevant                100
sufficient                100
syntactic_incomplete      100
Name: count, dtype: int64

Unique studies: 100

Frontal images: 96

Benchmark SHA-256:
2f73429a2620296d0a7a83454f9ca071ac9a38e829f28907ce27af101bc31488


In [ ]:
EXCLUDED_NO_FRONTAL = {74, 597, 803, 885}

eligible = df[~df["uid"].isin(EXCLUDED_NO_FRONTAL)].copy()

print("Eligible records:", len(eligible))
print("Eligible studies:", eligible["uid"].nunique())

print("\nRecords per condition:")
print(eligible["condition"].value_counts().sort_index())

print("\nExcluded records:")
print(df[df["uid"].isin(EXCLUDED_NO_FRONTAL)][
    ["uid", "condition", "frontal_image"]
].to_string(index=False))

Eligible records: 576
Eligible studies: 96

Records per condition:
condition
conflicting               96
evidentiary_incomplete    96
insufficient              96
irrelevant                96
sufficient                96
syntactic_incomplete      96
Name: count, dtype: int64

Excluded records:
 uid              condition frontal_image
  74             sufficient           NaN
  74   syntactic_incomplete           NaN
  74 evidentiary_incomplete           NaN
  74             irrelevant           NaN
  74            conflicting           NaN
  74           insufficient           NaN
 597             sufficient           NaN
 597   syntactic_incomplete           NaN
 597 evidentiary_incomplete           NaN
 597             irrelevant           NaN
 597            conflicting           NaN
 597           insufficient           NaN
 803             sufficient           NaN
 803   syntactic_incomplete           NaN
 803 evidentiary_incomplete           NaN
 803             irrelevant     

In [ ]:
DEV_STUDIES = [
    195, 227, 253, 282,
    290, 317, 458, 494,
    577, 783, 793, 936,
    960, 965, 1025, 1165
]

dev = eligible[eligible["uid"].isin(DEV_STUDIES)].copy()
evaluation = eligible[~eligible["uid"].isin(DEV_STUDIES)].copy()

print("Development:")
print("  records:", len(dev))
print("  studies:", dev["uid"].nunique())

print("\nDevelopment condition counts:")
print(dev["condition"].value_counts().sort_index())

print("\nEvaluation:")
print("  records:", len(evaluation))
print("  studies:", evaluation["uid"].nunique())

print("\nEvaluation condition counts:")
print(evaluation["condition"].value_counts().sort_index())

print("\nStudy overlap:")
print(set(dev["uid"]) & set(evaluation["uid"]))

Development:
  records: 96
  studies: 16

Development condition counts:
condition
conflicting               16
evidentiary_incomplete    16
insufficient              16
irrelevant                16
sufficient                16
syntactic_incomplete      16
Name: count, dtype: int64

Evaluation:
  records: 480
  studies: 80

Evaluation condition counts:
condition
conflicting               80
evidentiary_incomplete    80
insufficient              80
irrelevant                80
sufficient                80
syntactic_incomplete      80
Name: count, dtype: int64

Study overlap:
set()


In [ ]:
SPLIT_DIR = ROOT / "results" / "tables" / "phase26d"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

split_rows = []

for uid in sorted(eligible["uid"].unique()):
    split_rows.append({
        "study_id": int(uid),
        "split": "development" if uid in DEV_STUDIES else "evaluation"
    })

split_df = pd.DataFrame(split_rows)

SPLIT_PATH = SPLIT_DIR / "phase26d_maira2_split.csv"
split_df.to_csv(SPLIT_PATH, index=False)

print("Saved:")
print(SPLIT_PATH)
print("\nShape:", split_df.shape)
print(split_df["split"].value_counts())

Saved:
/content/Evidence-State-Radiology/results/tables/phase26d/phase26d_maira2_split.csv

Shape: (96, 2)
split
evaluation     80
development    16
Name: count, dtype: int64


In [ ]:
from pathlib import Path
import pandas as pd

SPLIT_PATH = Path(
    "/content/Evidence-State-Radiology/"
    "results/tables/phase26d/phase26d_maira2_split.csv"
)

split = pd.read_csv(SPLIT_PATH)

print(split)
print("\nSHA-256:")
import hashlib

h = hashlib.sha256()
with open(SPLIT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)

print(h.hexdigest())

    study_id        split
0         32   evaluation
1         36   evaluation
2         44   evaluation
3         49   evaluation
4         53   evaluation
..       ...          ...
91      1114   evaluation
92      1165  development
93      1172   evaluation
94      1173   evaluation
95      1205   evaluation

[96 rows x 2 columns]

SHA-256:
b7e37147cc59e6477f5d93ab9c826a967ac75c0883b014693197cf7a1d07a8aa


In [ ]:
!pip install -q \
    "transformers==4.51.3" \
    "accelerate>=1.2,<2" \
    "pillow" \
    "protobuf" \
    "sentencepiece" \
    "bitsandbytes"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
import transformers

print("Transformers:", transformers.__version__)

Transformers: 4.51.3


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token available:", HF_TOKEN is not None)

HF token available: True


In [ ]:
from transformers import AutoProcessor

MODEL_ID = "microsoft/maira-2"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)

print("Processor loaded:", type(processor).__name__)

processor_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

processing_maira2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/maira-2:
- processing_maira2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


preprocessor_config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Processor loaded: Maira2Processor


In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map={"": 0},
    quantization_config=quantization_config,
    token=HF_TOKEN,
)

model.eval()

print("Model class:", type(model).__name__)
print("Model loaded successfully.")

config.json: 0.00B [00:00, ?B/s]

configuration_maira2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/maira-2:
- configuration_maira2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_maira2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/maira-2:
- modeling_maira2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00001-of-00006.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00004-of-00006.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

Model class: Maira2ForConditionalGeneration
Model loaded successfully.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/Evidence-State-Radiology")

BENCHMARK = ROOT / "results" / "tables" / "phase16_perturbations.csv"

benchmark = pd.read_csv(BENCHMARK)

uid = 195

rows = benchmark[benchmark["uid"] == uid].copy()

print(rows[
    [
        "sample_id",
        "uid",
        "condition",
        "perturbed_context",
        "frontal_image",
        "findings",
        "impression",
    ]
].to_string(index=False))

sample_id  uid              condition                                perturbed_context                                   frontal_image                                                                              findings                                impression
   IU_195  195             sufficient XXXX-year-old female with coughing and wheezing. external_data/IU-Xray/images/IU_195_frontal.jpg Clear lungs bilaterally. No pneumothorax or pleural effusion. Normal cardiac contours 1. No acute cardiopulmonary abnormalities
   IU_195  195   syntactic_incomplete                        XXXX-year-old female with external_data/IU-Xray/images/IU_195_frontal.jpg Clear lungs bilaterally. No pneumothorax or pleural effusion. Normal cardiac contours 1. No acute cardiopulmonary abnormalities
   IU_195  195 evidentiary_incomplete                  XXXX-year-old female, wheezing. external_data/IU-Xray/images/IU_195_frontal.jpg Clear lungs bilaterally. No pneumothorax or pleural effusion. Normal cardiac

In [ ]:
uid195 = benchmark[benchmark["uid"] == 195].iloc[0]

print("Frontal image reference:")
print(uid195["frontal_image"])

print("\nImage exists locally:")

image_ref = str(uid195["frontal_image"])

candidates = [
    ROOT / image_ref,
    ROOT / "external_data" / "IU-Xray" / image_ref,
    Path("/content") / image_ref,
]

for p in candidates:
    print(p, "->", p.exists())

Frontal image reference:
external_data/IU-Xray/images/IU_195_frontal.jpg

Image exists locally:
/content/Evidence-State-Radiology/external_data/IU-Xray/images/IU_195_frontal.jpg -> False
/content/Evidence-State-Radiology/external_data/IU-Xray/external_data/IU-Xray/images/IU_195_frontal.jpg -> False
/content/external_data/IU-Xray/images/IU_195_frontal.jpg -> False


In [ ]:
from pathlib import Path

for p in [
    Path("/content/IU-Xray-source"),
    Path("/content/IU-Xray"),
    Path("/content"),
]:
    print("\n", p)
    if p.exists():
        matches = list(p.rglob("*.parquet"))
        print("Parquet files:", len(matches))
        for x in matches[:10]:
            print(" ", x)


 /content/IU-Xray-source

 /content/IU-Xray

 /content
Parquet files: 0


In [ ]:
from pathlib import Path

SOURCE_ROOT = Path("/content/IU-Xray-source")
DATA_DIR = SOURCE_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)

/content/IU-Xray-source/data


In [ ]:
!wget -q --show-progress \
"https://huggingface.co/datasets/ykumards/open-i/resolve/main/data/train-00000-of-00005-c80629f6027d8eb1.parquet" \
-O /content/IU-Xray-source/data/train-00000-of-00005-c80629f6027d8eb1.parquet

!wget -q --show-progress \
"https://huggingface.co/datasets/ykumards/open-i/resolve/main/data/train-00001-of-00005-07e6c158a3f56170.parquet" \
-O /content/IU-Xray-source/data/train-00001-of-00005-07e6c158a3f56170.parquet

!wget -q --show-progress \
"https://huggingface.co/datasets/ykumards/open-i/resolve/main/data/train-00002-of-00005-653f05c8b15c22c6.parquet" \
-O /content/IU-Xray-source/data/train-00002-of-00005-653f05c8b15c22c6.parquet

!wget -q --show-progress \
"https://huggingface.co/datasets/ykumards/open-i/resolve/main/data/train-00003-of-00005-1d66a487a8489b7f.parquet" \
-O /content/IU-Xray-source/data/train-00003-of-00005-1d66a487a8489b7f.parquet

!wget -q --show-progress \
"https://huggingface.co/datasets/ykumards/open-i/resolve/main/data/train-00004-of-00005-208b7d689f1c7b70.parquet" \
-O /content/IU-Xray-source/data/train-00004-of-00005-208b7d689f1c7b70.parquet

/content/IU-Xray-so 100%[===================>] 405.48M  20.2MB/s    in 7.0s    
/content/IU-Xray-so 100%[===================>] 398.66M  46.2MB/s    in 6.9s    
/content/IU-Xray-so 100%[===================>] 396.41M  43.9MB/s    in 10s     
/content/IU-Xray-so 100%[===================>] 396.53M   112MB/s    in 4.5s    
/content/IU-Xray-so 100%[===================>] 401.69M  89.9MB/s    in 5.1s    


In [ ]:
from pathlib import Path

parquets = sorted(DATA_DIR.glob("*.parquet"))

print("Parquet files:", len(parquets))

for p in parquets:
    print(f"{p.name}: {p.stat().st_size:,} bytes")

Parquet files: 5
train-00000-of-00005-c80629f6027d8eb1.parquet: 425,182,167 bytes
train-00001-of-00005-07e6c158a3f56170.parquet: 418,023,741 bytes
train-00002-of-00005-653f05c8b15c22c6.parquet: 415,669,328 bytes
train-00003-of-00005-1d66a487a8489b7f.parquet: 415,787,300 bytes
train-00004-of-00005-208b7d689f1c7b70.parquet: 421,207,075 bytes


In [ ]:
import pandas as pd
from PIL import Image
from io import BytesIO

TARGET_UID = 195
found = False

for parquet in parquets:
    print("Checking:", parquet.name)

    table = pd.read_parquet(
        parquet,
        columns=["uid", "img_frontal"]
    )

    matches = table[table["uid"] == TARGET_UID]

    if len(matches) == 0:
        continue

    row = matches.iloc[0]
    image_bytes = row["img_frontal"]

    print("Found UID:", TARGET_UID)
    print("Image bytes:", len(image_bytes))

    output_dir = ROOT / "external_data" / "IU-Xray" / "images"
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / "IU_195_frontal.jpg"

    with open(output_path, "wb") as f:
        f.write(image_bytes)

    print("Saved:", output_path)

    with Image.open(output_path) as img:
        print("Format:", img.format)
        print("Dimensions:", img.size)
        print("Mode:", img.mode)

    found = True
    break

if not found:
    raise RuntimeError("UID 195 was not found in any Parquet shard.")

Checking: train-00000-of-00005-c80629f6027d8eb1.parquet
Found UID: 195
Image bytes: 382593
Saved: /content/Evidence-State-Radiology/external_data/IU-Xray/images/IU_195_frontal.jpg
Format: JPEG
Dimensions: (2048, 2059)
Mode: L


In [ ]:
image_path = (
    ROOT
    / "external_data"
    / "IU-Xray"
    / "images"
    / "IU_195_frontal.jpg"
)

print("Exists:", image_path.exists())
print("Size:", image_path.stat().st_size, "bytes")

Exists: True
Size: 382593 bytes


In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd

ROOT = Path("/content/Evidence-State-Radiology")

benchmark = pd.read_csv(
    ROOT / "results" / "tables" / "phase16_perturbations.csv"
)

uid195_rows = benchmark[benchmark["uid"] == 195].copy()

image_path = ROOT / "external_data" / "IU-Xray" / "images" / "IU_195_frontal.jpg"

frontal_image = Image.open(image_path).convert("RGB")

print("Image:", image_path)
print("Image size:", frontal_image.size)
print()

print(
    uid195_rows[
        ["sample_id", "condition", "perturbed_context"]
    ].to_string(index=False)
)

Image: /content/Evidence-State-Radiology/external_data/IU-Xray/images/IU_195_frontal.jpg
Image size: (2048, 2059)

sample_id              condition                                perturbed_context
   IU_195             sufficient XXXX-year-old female with coughing and wheezing.
   IU_195   syntactic_incomplete                        XXXX-year-old female with
   IU_195 evidentiary_incomplete                  XXXX-year-old female, wheezing.
   IU_195             irrelevant                              Chest pain, dyspnea
   IU_195            conflicting                  The patient has a pneumothorax.
   IU_195           insufficient                                              NaN


In [ ]:
import torch

conflict_context = uid195_rows.loc[
    uid195_rows["condition"] == "conflicting",
    "perturbed_context"
].iloc[0]

print("Conflict context:")
print(conflict_context)

processed_context = processor.format_and_preprocess_reporting_input(
    current_frontal=frontal_image,
    current_lateral=None,
    prior_frontal=None,
    indication=conflict_context,
    technique=None,
    comparison=None,
    prior_report=None,
    return_tensors="pt",
    get_grounding=False,
).to("cuda")

processed_visual = processor.format_and_preprocess_reporting_input(
    current_frontal=frontal_image,
    current_lateral=None,
    prior_frontal=None,
    indication=None,
    technique=None,
    comparison=None,
    prior_report=None,
    return_tensors="pt",
    get_grounding=False,
).to("cuda")

print("Context input:")
for k, v in processed_context.items():
    if hasattr(v, "shape"):
        print(k, tuple(v.shape))

print("\nVisual input:")
for k, v in processed_visual.items():
    if hasattr(v, "shape"):
        print(k, tuple(v.shape))

Conflict context:
The patient has a pneumothorax.
Context input:
input_ids (1, 1462)
attention_mask (1, 1462)
pixel_values (1, 3, 518, 518)

Visual input:
input_ids (1, 1455)
attention_mask (1, 1455)
pixel_values (1, 3, 518, 518)


In [ ]:
with torch.inference_mode():
    official_output = model.generate(
        **processed_context,
        max_new_tokens=128,
        use_cache=True,
    )

official_text = processor.decode(
    official_output[0],
    skip_special_tokens=True
)

print("OFFICIAL MAIRA-2")
print(official_text)
print("Token count:", official_output.shape[1])

OFFICIAL MAIRA-2
You are an expert radiology assistant tasked with interpreting a chest X-ray study.  USER:  Given the current frontal image only<image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><

In [ ]:
with torch.inference_mode():
    context_forward = model(
        **processed_context,
        use_cache=True,
    )

    visual_forward = model(
        **processed_visual,
        use_cache=True,
    )

context_cache = context_forward.past_key_values
visual_cache = visual_forward.past_key_values

context_logits = context_forward.logits[:, -1, :]
visual_logits = visual_forward.logits[:, -1, :]

print("Context cache:", type(context_cache).__name__)
print("Visual cache:", type(visual_cache).__name__)

print("Context logits:", tuple(context_logits.shape))
print("Visual logits:", tuple(visual_logits.shape))

Context cache: DynamicCache
Visual cache: DynamicCache
Context logits: (1, 32207)
Visual logits: (1, 32207)


In [ ]:
import torch

def two_stream_decode_no_copy(
    model,
    context_cache,
    visual_cache,
    context_logits,
    visual_logits,
    lambda_value=0.0,
    max_new_tokens=128,
):
    """
    Two-stream contrastive decoding for MAIRA-2.

    Context stream:
        image + clinical context

    Visual stream:
        image only

    At every generation step:
        adjusted_logits =
            context_logits
            - lambda * (context_logits - visual_logits)

    The selected token is appended to both streams.

    lambda=0 must exactly reproduce ordinary
    context-conditioned MAIRA-2 greedy generation.
    """

    device = context_logits.device

    # First generated token from the initial prompt caches.
    adjusted_logits = (
        context_logits
        - lambda_value * (context_logits - visual_logits)
    )

    next_token = torch.argmax(
        adjusted_logits,
        dim=-1,
        keepdim=True,
    )

    generated_tokens = [next_token]

    # Continue autoregressively.
    for _ in range(max_new_tokens - 1):

        # Feed the previously selected token into BOTH streams.
        context_outputs = model(
            input_ids=next_token,
            past_key_values=context_cache,
            use_cache=True,
        )

        visual_outputs = model(
            input_ids=next_token,
            past_key_values=visual_cache,
            use_cache=True,
        )

        context_cache = context_outputs.past_key_values
        visual_cache = visual_outputs.past_key_values

        context_logits = context_outputs.logits[:, -1, :]
        visual_logits = visual_outputs.logits[:, -1, :]

        adjusted_logits = (
            context_logits
            - lambda_value * (
                context_logits - visual_logits
            )
        )

        next_token = torch.argmax(
            adjusted_logits,
            dim=-1,
            keepdim=True,
        )

        generated_tokens.append(next_token)

    return torch.cat(generated_tokens, dim=1)

In [ ]:
with torch.inference_mode():
    context_forward = model(
        **processed_context,
        use_cache=True,
    )

    visual_forward = model(
        **processed_visual,
        use_cache=True,
    )

context_cache_test = context_forward.past_key_values
visual_cache_test = visual_forward.past_key_values

context_logits_test = context_forward.logits[:, -1, :]
visual_logits_test = visual_forward.logits[:, -1, :]

print("Fresh caches ready.")
print("Context cache:", type(context_cache_test).__name__)
print("Visual cache:", type(visual_cache_test).__name__)

Fresh caches ready.
Context cache: DynamicCache
Visual cache: DynamicCache


In [ ]:
with torch.inference_mode():
    custom_output = two_stream_decode_no_copy(
        model=model,
        context_cache=context_cache_test,
        visual_cache=visual_cache_test,
        context_logits=context_logits_test,
        visual_logits=visual_logits_test,
        lambda_value=0.0,
        max_new_tokens=128,
    )

custom_text = processor.decode(
    custom_output[0],
    skip_special_tokens=True,
)

print("CUSTOM TWO-STREAM λ=0")
print(custom_text)

print()
print("Official token shape:", tuple(official_output.shape))
print("Custom token shape:  ", tuple(custom_output.shape))

print(
    "Exact token equivalence:",
    torch.equal(official_output, custom_output)
)

print(
    "Exact text equivalence:",
    official_text == custom_text
)

CUSTOM TWO-STREAM λ=0
A single-view chest x-ray demonstrates a normal heart size. There is no acute consolidation, effusion or pneumothorax seen. No adenopathy is appreciated.ss a small right apical pneumothorax. This finding was not seen on the prior examination.s a small right apical pneumothorax. This finding was not seen on the prior examination.s a small right apical pneumothorax. This finding was not seen on the prior examination.s a small right apical pneum

Official token shape: (1, 1507)
Custom token shape:   (1, 128)
Exact token equivalence: False
Exact text equivalence: False


In [ ]:
context_prompt_len = processed_context["input_ids"].shape[1]

official_generated = official_output[
    :,
    context_prompt_len:
]

print("Prompt tokens:", context_prompt_len)
print("Official total tokens:", official_output.shape[1])
print("Official generated tokens:", official_generated.shape[1])

print(
    "Official EOS token:",
    model.generation_config.eos_token_id
)

print(
    "Last official generated token:",
    official_generated[0, -1].item()
)

Prompt tokens: 1462
Official total tokens: 1507
Official generated tokens: 45
Official EOS token: 2
Last official generated token: 2


In [ ]:
eos_id = model.generation_config.eos_token_id

eos_positions = (
    official_generated[0] == eos_id
).nonzero(as_tuple=True)[0]

print("EOS positions:", eos_positions.tolist())

EOS positions: [44]


In [ ]:
n = min(
    official_generated.shape[1],
    custom_output.shape[1]
)

official_prefix = official_generated[0, :n]
custom_prefix = custom_output[0, :n]

matches = official_prefix == custom_prefix

print("Compared tokens:", n)
print("Matching tokens:", matches.sum().item())
print("First mismatch:",
      (~matches).nonzero(as_tuple=True)[0][0].item()
      if (~matches).any() else "NONE"
)

Compared tokens: 45
Matching tokens: 45
First mismatch: NONE


In [ ]:
print("\nOFFICIAL GENERATED:")
print(
    processor.decode(
        official_generated[0, :60],
        skip_special_tokens=False
    )
)

print("\nCUSTOM:")
print(
    processor.decode(
        custom_output[0, :60],
        skip_special_tokens=False
    )
)


OFFICIAL GENERATED:
A single-view chest x-ray demonstrates a normal heart size. There is no acute consolidation, effusion or pneumothorax seen. No adenopathy is appreciated.</s>

CUSTOM:
A single-view chest x-ray demonstrates a normal heart size. There is no acute consolidation, effusion or pneumothorax seen. No adenopathy is appreciated.</s>s</s></s>s a small right apical pneumothorax.


In [ ]:
print("Official first generated token:")
print(
    official_generated[0, 0].item(),
    repr(
        processor.decode(
            official_generated[0, :1],
            skip_special_tokens=False
        )
    )
)

print("\nCustom first generated token:")
print(
    custom_output[0, 0].item(),
    repr(
        processor.decode(
            custom_output[0, :1],
            skip_special_tokens=False
        )
    )
)

Official first generated token:
319 'A'

Custom first generated token:
319 'A'


In [ ]:
import torch

def two_stream_decode_no_copy(
    model,
    context_cache,
    visual_cache,
    context_logits,
    visual_logits,
    lambda_value=0.0,
    max_new_tokens=128,
):
    """
    MAIRA-2 two-stream contrastive greedy decoder.

    Context stream:
        image + clinical context

    Visual stream:
        image only

    adjusted_logits =
        context_logits
        - lambda * (context_logits - visual_logits)

    lambda=0 reproduces the context-conditioned
    MAIRA-2 greedy generation sequence.

    Stops immediately at EOS.
    """

    eos_id = model.generation_config.eos_token_id

    generated_tokens = []

    # First token
    adjusted_logits = (
        context_logits
        - lambda_value * (context_logits - visual_logits)
    )

    next_token = torch.argmax(
        adjusted_logits,
        dim=-1,
        keepdim=True,
    )

    generated_tokens.append(next_token)

    # Stop immediately if first token is EOS
    if eos_id is not None and torch.all(next_token == eos_id):
        return torch.cat(generated_tokens, dim=1)

    for _ in range(max_new_tokens - 1):

        context_outputs = model(
            input_ids=next_token,
            past_key_values=context_cache,
            use_cache=True,
        )

        visual_outputs = model(
            input_ids=next_token,
            past_key_values=visual_cache,
            use_cache=True,
        )

        context_cache = context_outputs.past_key_values
        visual_cache = visual_outputs.past_key_values

        context_logits = context_outputs.logits[:, -1, :]
        visual_logits = visual_outputs.logits[:, -1, :]

        adjusted_logits = (
            context_logits
            - lambda_value * (
                context_logits - visual_logits
            )
        )

        next_token = torch.argmax(
            adjusted_logits,
            dim=-1,
            keepdim=True,
        )

        generated_tokens.append(next_token)

        # MAIRA-2 EOS termination
        if eos_id is not None and torch.all(next_token == eos_id):
            break

    return torch.cat(generated_tokens, dim=1)

In [ ]:
with torch.inference_mode():
    context_forward = model(
        **processed_context,
        use_cache=True,
    )

    visual_forward = model(
        **processed_visual,
        use_cache=True,
    )

context_cache_test = context_forward.past_key_values
visual_cache_test = visual_forward.past_key_values

context_logits_test = context_forward.logits[:, -1, :]
visual_logits_test = visual_forward.logits[:, -1, :]

In [ ]:
with torch.inference_mode():
    custom_output = two_stream_decode_no_copy(
        model=model,
        context_cache=context_cache_test,
        visual_cache=visual_cache_test,
        context_logits=context_logits_test,
        visual_logits=visual_logits_test,
        lambda_value=0.0,
        max_new_tokens=128,
    )

custom_text = processor.decode(
    custom_output[0],
    skip_special_tokens=True,
)

official_generated = official_output[
    :,
    processed_context["input_ids"].shape[1]:
]

print("Official generated shape:", tuple(official_generated.shape))
print("Custom generated shape:  ", tuple(custom_output.shape))

print("\nOfficial:")
print(
    processor.decode(
        official_generated[0],
        skip_special_tokens=False
    )
)

print("\nCustom:")
print(
    processor.decode(
        custom_output[0],
        skip_special_tokens=False
    )
)

print("\nExact token equivalence:",
      torch.equal(official_generated, custom_output))

print("Exact text equivalence:",
      processor.decode(
          official_generated[0],
          skip_special_tokens=True
      ) == custom_text)

Official generated shape: (1, 45)
Custom generated shape:   (1, 45)

Official:
A single-view chest x-ray demonstrates a normal heart size. There is no acute consolidation, effusion or pneumothorax seen. No adenopathy is appreciated.</s>

Custom:
A single-view chest x-ray demonstrates a normal heart size. There is no acute consolidation, effusion or pneumothorax seen. No adenopathy is appreciated.</s>

Exact token equivalence: True
Exact text equivalence: True


In [ ]:
from pathlib import Path
import pandas as pd
import hashlib

ROOT = Path("/content/Evidence-State-Radiology")

OUT_DIR = ROOT / "results" / "tables" / "phase26d_claim_evaluation"
DOC_DIR = ROOT / "docs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

lambda_rows = [
    {"lambda": 0.00, "unsupported_rate": 0.076087, "supported_claims": 96, "omissions": 124},
    {"lambda": 0.10, "unsupported_rate": 0.077348, "supported_claims": 89, "omissions": 128},
    {"lambda": 0.25, "unsupported_rate": 0.060440, "supported_claims": 84, "omissions": 132},
    {"lambda": 0.40, "unsupported_rate": 0.048485, "supported_claims": 67, "omissions": 152},
    {"lambda": 0.50, "unsupported_rate": 0.021739, "supported_claims": 61, "omissions": 163},
]

df = pd.DataFrame(lambda_rows)

baseline_supported = 96
baseline_omissions = 124
baseline_risk = 0.076087

df["supported_retention"] = df["supported_claims"] / baseline_supported
df["omission_increase"] = (
    (df["omissions"] - baseline_omissions) / baseline_omissions
)
df["risk_reduction"] = (
    (baseline_risk - df["unsupported_rate"]) / baseline_risk
)

df["qualifies"] = (
    (df["risk_reduction"] > 0)
    & (df["supported_retention"] >= 0.80)
    & (df["omission_increase"] <= 0.25)
)

eligible = df[df["qualifies"]].sort_values("lambda")

selected_lambda = float(eligible.iloc[0]["lambda"])

print(df.to_string(index=False))
print("\nSELECTED LAMBDA:", selected_lambda)

assert selected_lambda == 0.25

 lambda  unsupported_rate  supported_claims  omissions  supported_retention  omission_increase  risk_reduction  qualifies
   0.00          0.076087                96        124             1.000000           0.000000        0.000000      False
   0.10          0.077348                89        128             0.927083           0.032258       -0.016573      False
   0.25          0.060440                84        132             0.875000           0.064516        0.205646       True
   0.40          0.048485                67        152             0.697917           0.225806        0.362769      False
   0.50          0.021739                61        163             0.635417           0.314516        0.714288      False

SELECTED LAMBDA: 0.25


In [ ]:
policy_csv = OUT_DIR / "phase26d_lambda_selection.csv"
df.to_csv(policy_csv, index=False)

policy_text = f"""# Phase 26D-7 — Lambda Selection Policy

## Purpose

Select the contrastive decoding strength using the development cohort
before evaluation-cohort generation.

## Development cohort

- 16 studies
- 96 condition-level records
- 6 evidence conditions
- Lambda candidates: 0.00, 0.10, 0.25, 0.40, 0.50

## Selection rule

1. Primary objective: reduce pooled unsupported claim rate relative to lambda=0.
2. Supported-claim retention must be >= 80% of lambda=0.
3. Omission increase must be <= 25% relative to lambda=0.
4. Among candidates satisfying all constraints, select the smallest lambda.

## Frozen selection

Selected lambda = {selected_lambda:.2f}

The evaluation cohort was not used to select lambda.
"""

policy_md = DOC_DIR / "phase26d_lambda_selection_policy.md"
policy_md.write_text(policy_text, encoding="utf-8")

print(policy_md)
print(policy_csv)

/content/Evidence-State-Radiology/docs/phase26d_lambda_selection_policy.md
/content/Evidence-State-Radiology/results/tables/phase26d_claim_evaluation/phase26d_lambda_selection.csv


In [ ]:
print("Split columns:")
print(split.columns.tolist())

print("\nFirst rows:")
print(split.head().to_string(index=False))

Split columns:
['study_id', 'split']

First rows:
 study_id      split
       32 evaluation
       36 evaluation
       44 evaluation
       49 evaluation
       53 evaluation


In [ ]:
# The frozen split is study-level:
# study_id -> split
#
# The benchmark is condition-level:
# uid/sample_id/condition -> benchmark row

split_check = benchmark.copy()

split_check["study_id"] = split_check["uid"].astype(int)

split_check = split_check.merge(
    split,
    on="study_id",
    how="left",
    validate="many_to_one",
)

print("Rows:", len(split_check))
print("Missing split:", split_check["split"].isna().sum())

print("\nSplit × condition:")
print(
    split_check
    .groupby(["split", "condition"])
    .size()
    .unstack(fill_value=0)
)

print("\nStudies by split:")
print(
    split_check
    .groupby("split")["uid"]
    .nunique()
)

assert len(split_check) == 600
assert split_check["split"].notna().all()

dev = split_check[split_check["split"] == "development"]
evaluation = split_check[split_check["split"] == "evaluation"]

assert dev["uid"].nunique() == 16
assert evaluation["uid"].nunique() == 80

assert len(dev) == 96
assert len(evaluation) == 480

assert dev.groupby("condition").size().eq(16).all()
assert evaluation.groupby("condition").size().eq(80).all()

assert set(dev["uid"]).isdisjoint(
    set(evaluation["uid"])
)

print("\nPHASE 26D SPLIT VALIDATION: PASS")

Rows: 600
Missing split: 24

Split × condition:
condition    conflicting  evidentiary_incomplete  insufficient  irrelevant  \
split                                                                        
development           16                      16            16          16   
evaluation            80                      80            80          80   

condition    sufficient  syntactic_incomplete  
split                                          
development          16                    16  
evaluation           80                    80  

Studies by split:
split
development    16
evaluation     80
Name: uid, dtype: int64


AssertionError: 

In [ ]:
EXCLUDED_UIDS = {74, 597, 803, 885}

eligible_benchmark = benchmark[
    ~benchmark["uid"].astype(int).isin(EXCLUDED_UIDS)
].copy()

eligible_benchmark["study_id"] = (
    eligible_benchmark["uid"].astype(int)
)

split_check = eligible_benchmark.merge(
    split,
    on="study_id",
    how="left",
    validate="many_to_one",
)

print("Eligible benchmark rows:", len(eligible_benchmark))
print("Split rows:", len(split_check))
print(
    "Missing split:",
    split_check["split"].isna().sum()
)

print("\nSplit × condition:")
print(
    split_check
    .groupby(["split", "condition"])
    .size()
    .unstack(fill_value=0)
)

print("\nStudies by split:")
print(
    split_check
    .groupby("split")["uid"]
    .nunique()
)

assert len(eligible_benchmark) == 576
assert len(split_check) == 576
assert split_check["split"].notna().all()

dev = split_check[
    split_check["split"] == "development"
]

evaluation = split_check[
    split_check["split"] == "evaluation"
]

assert dev["uid"].nunique() == 16
assert evaluation["uid"].nunique() == 80

assert len(dev) == 96
assert len(evaluation) == 480

assert (
    dev.groupby("condition")
    .size()
    .eq(16)
    .all()
)

assert (
    evaluation.groupby("condition")
    .size()
    .eq(80)
    .all()
)

assert set(dev["uid"]).isdisjoint(
    set(evaluation["uid"])
)

print("\nPHASE 26D SPLIT VALIDATION: PASS")

Eligible benchmark rows: 576
Split rows: 576
Missing split: 0

Split × condition:
condition    conflicting  evidentiary_incomplete  insufficient  irrelevant  \
split                                                                        
development           16                      16            16          16   
evaluation            80                      80            80          80   

condition    sufficient  syntactic_incomplete  
split                                          
development          16                    16  
evaluation           80                    80  

Studies by split:
split
development    16
evaluation     80
Name: uid, dtype: int64

PHASE 26D SPLIT VALIDATION: PASS


In [ ]:
eval_benchmark = split_check[
    split_check["split"] == "evaluation"
].copy()

eval_benchmark = eval_benchmark.sort_values(
    ["uid", "condition"]
).reset_index(drop=True)

print("Evaluation records:", len(eval_benchmark))
print("Evaluation studies:", eval_benchmark["uid"].nunique())

print("\nCondition counts:")
print(
    eval_benchmark["condition"]
    .value_counts()
    .sort_index()
)

assert len(eval_benchmark) == 480
assert eval_benchmark["uid"].nunique() == 80
assert (
    eval_benchmark["condition"]
    .value_counts()
    .eq(80)
    .all()
)

print("\nEVALUATION COHORT: PASS")

Evaluation records: 480
Evaluation studies: 80

Condition counts:
condition
conflicting               80
evidentiary_incomplete    80
insufficient              80
irrelevant                80
sufficient                80
syntactic_incomplete      80
Name: count, dtype: int64

EVALUATION COHORT: PASS


In [ ]:
from pathlib import Path
import pandas as pd

eval_uids = sorted(
    eval_benchmark["uid"].astype(int).unique()
)

image_dir = (
    ROOT
    / "external_data"
    / "IU-Xray"
    / "images"
)

image_dir.mkdir(parents=True, exist_ok=True)

remaining = set(eval_uids)

for parquet in sorted(DATA_DIR.glob("*.parquet")):

    if not remaining:
        break

    print("\nChecking:", parquet.name)

    table = pd.read_parquet(
        parquet,
        columns=["uid", "img_frontal"]
    )

    matches = table[
        table["uid"].isin(remaining)
    ]

    print("Matches:", len(matches))

    for _, row in matches.iterrows():

        uid = int(row["uid"])
        image_bytes = row["img_frontal"]

        if image_bytes is None:
            print("WARNING: missing frontal:", uid)
            continue

        output_path = (
            image_dir / f"IU_{uid}_frontal.jpg"
        )

        with open(output_path, "wb") as f:
            f.write(image_bytes)

        remaining.remove(uid)

print("\nExpected:", len(eval_uids))
print("Recovered:", len(eval_uids) - len(remaining))
print("Missing:", sorted(remaining))

assert len(remaining) == 0

print("\nALL 80 EVALUATION IMAGES RECOVERED: PASS")


Checking: train-00000-of-00005-c80629f6027d8eb1.parquet
Matches: 61

Checking: train-00001-of-00005-07e6c158a3f56170.parquet
Matches: 19

Expected: 80
Recovered: 80
Missing: []

ALL 80 EVALUATION IMAGES RECOVERED: PASS


In [ ]:
from PIL import Image

audit = []

for uid in eval_uids:

    path = image_dir / f"IU_{uid}_frontal.jpg"

    assert path.exists(), f"Missing: {uid}"

    with Image.open(path) as img:
        img.verify()

    with Image.open(path) as img:
        audit.append({
            "uid": uid,
            "bytes": path.stat().st_size,
            "width": img.width,
            "height": img.height,
        })

image_audit = pd.DataFrame(audit)

print(image_audit.to_string(index=False))

print("\nImages:", len(image_audit))
print("Unique UIDs:", image_audit["uid"].nunique())
print("Zero-byte:", (image_audit["bytes"] == 0).sum())

assert len(image_audit) == 80
assert image_audit["uid"].nunique() == 80
assert (image_audit["bytes"] > 0).all()

print("\nIMAGE INTEGRITY: PASS")

 uid  bytes  width  height
  32 214452   2051    2048
  36 300010   2048    2496
  44 268500   2048    2070
  49 302387   2496    2048
  53 211711   2048    2048
  66 343193   2121    2048
  67 298940   2048    2048
 175 237941   2075    2048
 185 268012   2048    2048
 190 244610   2326    2048
 247 269477   2048    2050
 254 346080   2048    2496
 270 328510   2048    2051
 275 314403   2048    2496
 287 460847   2494    2048
 297 379094   2494    2048
 303 362519   2131    2048
 304 459107   2494    2048
 313 269016   2048    2048
 329 276666   2486    2048
 340 422374   2494    2048
 341 353696   2048    2157
 372 547110   2494    2048
 382 301044   2048    2496
 388 282116   2048    2048
 400 302070   2496    2048
 417 418099   2455    2048
 437 390840   2496    2048
 451 305104   2048    2496
 460 218963   2048    2048
 461 291464   2048    2048
 463 306149   2048    2048
 475 263408   2051    2048
 486 335356   2048    2048
 493 282194   2496    2048
 498 378243   2496    2048
 

In [ ]:
import time
import pandas as pd
import torch
from pathlib import Path
from PIL import Image

EVAL_OUT = (
    ROOT
    / "results"
    / "maira2_contrastive"
    / "phase26d_evaluation"
)

EVAL_OUT.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_CSV = (
    EVAL_OUT
    / "phase26d_evaluation_generation.csv"
)

SELECTED_LAMBDA = 0.25

print("Output:", RESULT_CSV)
print("Selected lambda:", SELECTED_LAMBDA)

Output: /content/Evidence-State-Radiology/results/maira2_contrastive/phase26d_evaluation/phase26d_evaluation_generation.csv
Selected lambda: 0.25


In [ ]:
eval_benchmark = eval_benchmark.sort_values(
    ["uid", "condition"]
).reset_index(drop=True)

print("Evaluation records:", len(eval_benchmark))
print("Studies:", eval_benchmark["uid"].nunique())

print(
    eval_benchmark[
        ["sample_id", "uid", "condition", "perturbed_context"]
    ].head(12).to_string(index=False)
)

assert len(eval_benchmark) == 480
assert eval_benchmark["uid"].nunique() == 80
assert eval_benchmark["condition"].value_counts().eq(80).all()

Evaluation records: 480
Studies: 80
sample_id  uid              condition                                               perturbed_context
    IU_32   32            conflicting                                        Cardiomegaly is present.
    IU_32   32 evidentiary_incomplete                     hx XXXX nodules, XXXX for changes in lungs.
    IU_32   32           insufficient                                                             NaN
    IU_32   32             irrelevant          XXXX XXXX with COPD/emphysema and dyspnea on exertion.
    IU_32   32             sufficient WEAKNESS OF MUSCLES; hx XXXX nodules; XXXX for changes in lungs
    IU_32   32   syntactic_incomplete                                                     WEAKNESS OF
    IU_36   36            conflicting                                 The patient has a pneumothorax.
    IU_36   36 evidentiary_incomplete                       XXXX-year-old female, rule out pneumonia.
    IU_36   36           insufficient         

In [ ]:
def prepare_maira_inputs(image, context):

    context_input = processor.format_and_preprocess_reporting_input(
        current_frontal=image,
        current_lateral=None,
        prior_frontal=None,
        indication=context if pd.notna(context) else None,
        technique=None,
        comparison=None,
        prior_report=None,
        return_tensors="pt",
        get_grounding=False,
    ).to("cuda")

    visual_input = processor.format_and_preprocess_reporting_input(
        current_frontal=image,
        current_lateral=None,
        prior_frontal=None,
        indication=None,
        technique=None,
        comparison=None,
        prior_report=None,
        return_tensors="pt",
        get_grounding=False,
    ).to("cuda")

    return context_input, visual_input

In [ ]:
def generate_maira_trajectory(
    image,
    context,
    lambda_value,
):

    context_input, visual_input = prepare_maira_inputs(
        image,
        context,
    )

    with torch.inference_mode():

        context_forward = model(
            **context_input,
            use_cache=True,
        )

        visual_forward = model(
            **visual_input,
            use_cache=True,
        )

    context_cache = (
        context_forward.past_key_values
    )

    visual_cache = (
        visual_forward.past_key_values
    )

    context_logits = (
        context_forward.logits[:, -1, :]
    )

    visual_logits = (
        visual_forward.logits[:, -1, :]
    )

    with torch.inference_mode():

        output_tokens = two_stream_decode_no_copy(
            model=model,
            context_cache=context_cache,
            visual_cache=visual_cache,
            context_logits=context_logits,
            visual_logits=visual_logits,
            lambda_value=lambda_value,
            max_new_tokens=128,
        )

    text = processor.decode(
        output_tokens[0],
        skip_special_tokens=True,
    )

    return (
        text,
        int(output_tokens.shape[1]),
    )

In [ ]:
# ============================================================
# Phase26D Evaluation Generation
# Resume-safe checkpointed generation
# Key = (sample_id, condition, lambda)
# ============================================================

if RESULT_CSV.exists():

    completed = pd.read_csv(
        RESULT_CSV
    )

else:

    completed = pd.DataFrame()


# ------------------------------------------------------------
# Build completed trajectory keys
# IMPORTANT:
# condition MUST be part of the key.
# ------------------------------------------------------------

completed_keys = set()

if len(completed):

    completed_keys = set(
        zip(
            completed["sample_id"].astype(str),
            completed["condition"].astype(str),
            completed["lambda"].astype(float)
        )
    )


# ------------------------------------------------------------
# Preserve all existing checkpoint rows
# ------------------------------------------------------------

rows = (
    completed.to_dict("records")
    if len(completed)
    else []
)


# ------------------------------------------------------------
# Evaluation target
# 480 records × 2 lambdas = 960 trajectories
# ------------------------------------------------------------

total = len(eval_benchmark) * 2
processed = len(completed_keys)

print(
    f"Existing completed trajectories: "
    f"{processed}/{total}"
)

print(
    f"Remaining trajectories: "
    f"{total - processed}"
)


# ------------------------------------------------------------
# Generation loop
# ------------------------------------------------------------

for _, row in eval_benchmark.iterrows():

    sample_id = str(
        row["sample_id"]
    )

    condition = str(
        row["condition"]
    )

    image_path = (
        ROOT / row["frontal_image"]
    )

    image = Image.open(
        image_path
    ).convert("RGB")


    for lambda_value in [
        0.00,
        SELECTED_LAMBDA,
    ]:

        # ----------------------------------------------------
        # IMPORTANT:
        # A trajectory is uniquely identified by:
        #
        # (sample_id, condition, lambda)
        # ----------------------------------------------------

        key = (
            sample_id,
            condition,
            float(lambda_value),
        )


        # ----------------------------------------------------
        # Skip trajectories already present in checkpoint
        # ----------------------------------------------------

        if key in completed_keys:

            continue


        start = time.time()


        try:

            text, token_count = (
                generate_maira_trajectory(
                    image=image,
                    context=row[
                        "perturbed_context"
                    ],
                    lambda_value=lambda_value,
                )
            )

            status = "success"
            error = ""


        except Exception as exc:

            text = ""
            token_count = 0
            status = "failed"
            error = repr(exc)


        elapsed = (
            time.time() - start
        )


        # ----------------------------------------------------
        # Store trajectory result
        # ----------------------------------------------------

        record = {

            "sample_id": sample_id,

            "uid": int(
                row["uid"]
            ),

            "condition": condition,

            "evidence_state": row[
                "evidence_state"
            ],

            "perturbed_context": row[
                "perturbed_context"
            ],

            "frontal_image": row[
                "frontal_image"
            ],

            "findings": row[
                "findings"
            ],

            "impression": row[
                "impression"
            ],

            "lambda": float(
                lambda_value
            ),

            "report": text,

            "num_tokens": token_count,

            "runtime_seconds": elapsed,

            "status": status,

            "error": error,
        }


        rows.append(
            record
        )


        # ----------------------------------------------------
        # Mark this exact trajectory as completed
        # ----------------------------------------------------

        completed_keys.add(
            key
        )

        processed += 1


        # ----------------------------------------------------
        # Checkpoint after EVERY trajectory
        # ----------------------------------------------------

        pd.DataFrame(
            rows
        ).to_csv(
            RESULT_CSV,
            index=False,
        )


        print(
            f"[{processed}/{total}] "
            f"{sample_id} | "
            f"{condition} | "
            f"λ={lambda_value:.2f} | "
            f"{status} | "
            f"{elapsed:.2f}s"
        )


print("\n================================")
print("EVALUATION GENERATION COMPLETE")
print("Rows:", len(rows))
print("Output:", RESULT_CSV)
print("================================")

Existing completed trajectories: 160/960
Remaining trajectories: 800
[161/960] IU_32 | evidentiary_incomplete | λ=0.00 | success | 7.97s
[162/960] IU_32 | evidentiary_incomplete | λ=0.25 | success | 7.69s
[163/960] IU_32 | insufficient | λ=0.00 | success | 4.43s
[164/960] IU_32 | insufficient | λ=0.25 | success | 4.27s
[165/960] IU_32 | irrelevant | λ=0.00 | success | 7.94s
[166/960] IU_32 | irrelevant | λ=0.25 | success | 7.12s
[167/960] IU_32 | sufficient | λ=0.00 | success | 6.00s
[168/960] IU_32 | sufficient | λ=0.25 | success | 6.39s
[169/960] IU_32 | syntactic_incomplete | λ=0.00 | success | 3.89s
[170/960] IU_32 | syntactic_incomplete | λ=0.25 | success | 3.80s
[171/960] IU_36 | evidentiary_incomplete | λ=0.00 | success | 16.88s
[172/960] IU_36 | evidentiary_incomplete | λ=0.25 | success | 9.21s
[173/960] IU_36 | insufficient | λ=0.00 | success | 4.17s
[174/960] IU_36 | insufficient | λ=0.25 | success | 4.17s
[175/960] IU_36 | irrelevant | λ=0.00 | success | 7.91s
[176/960] IU_3

In [ ]:
# ============================================================
# Phase26D Evaluation Generation Audit
# ============================================================

import pandas as pd

eval_gen = pd.read_csv(
    RESULT_CSV
)

print("Rows:", len(eval_gen))
print("Expected rows: 960")

print("\nStudies:", eval_gen["uid"].nunique())
print("Expected studies: 80")

print("\nConditions:")
print(
    eval_gen["condition"]
    .value_counts()
    .sort_index()
)

print("\nLambda:")
print(
    eval_gen["lambda"]
    .value_counts()
    .sort_index()
)

print("\nStatus:")
print(
    eval_gen["status"]
    .value_counts()
)

# ------------------------------------------------------------
# Expected condition × lambda = 80 each
# ------------------------------------------------------------

condition_lambda_counts = (
    eval_gen
    .groupby(["condition", "lambda"])
    .size()
    .unstack(fill_value=0)
)

print("\nCondition × Lambda:")
print(condition_lambda_counts)

# ------------------------------------------------------------
# Duplicate trajectory audit
# ------------------------------------------------------------

duplicate_mask = (
    eval_gen
    .duplicated(
        subset=[
            "sample_id",
            "condition",
            "lambda"
        ],
        keep=False
    )
)

print(
    "\nDuplicate trajectory rows:",
    int(duplicate_mask.sum())
)

# ------------------------------------------------------------
# Missing trajectory audit
# ------------------------------------------------------------

expected_keys = {
    (
        str(row["sample_id"]),
        str(row["condition"]),
        float(lam)
    )
    for _, row in eval_benchmark.iterrows()
    for lam in [0.0, 0.25]
}

actual_keys = {
    (
        str(row["sample_id"]),
        str(row["condition"]),
        float(row["lambda"])
    )
    for _, row in eval_gen.iterrows()
}

missing_keys = expected_keys - actual_keys
extra_keys = actual_keys - expected_keys

print(
    "Missing trajectories:",
    len(missing_keys)
)

print(
    "Unexpected trajectories:",
    len(extra_keys)
)

# ------------------------------------------------------------
# Empty/failed output audit
# ------------------------------------------------------------

empty_reports = (
    eval_gen["report"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

print(
    "\nEmpty reports:",
    int(empty_reports.sum())
)

print(
    "Failed trajectories:",
    int(
        (eval_gen["status"] != "success").sum()
    )
)

# ------------------------------------------------------------
# Final PASS/FAIL
# ------------------------------------------------------------

audit_pass = (
    len(eval_gen) == 960
    and eval_gen["uid"].nunique() == 80
    and eval_gen["condition"].value_counts().eq(160).all()
    and eval_gen["lambda"].value_counts().eq(480).all()
    and int(duplicate_mask.sum()) == 0
    and len(missing_keys) == 0
    and len(extra_keys) == 0
    and int(empty_reports.sum()) == 0
    and int((eval_gen["status"] != "success").sum()) == 0
)

print("\n================================")
print(
    "PHASE26D EVALUATION GENERATION AUDIT:",
    "PASS" if audit_pass else "FAIL"
)
print("================================")

assert audit_pass

Rows: 960
Expected rows: 960

Studies: 80
Expected studies: 80

Conditions:
condition
conflicting               160
evidentiary_incomplete    160
insufficient              160
irrelevant                160
sufficient                160
syntactic_incomplete      160
Name: count, dtype: int64

Lambda:
lambda
0.00    480
0.25    480
Name: count, dtype: int64

Status:
status
success    960
Name: count, dtype: int64

Condition × Lambda:
lambda                  0.00  0.25
condition                         
conflicting               80    80
evidentiary_incomplete    80    80
insufficient              80    80
irrelevant                80    80
sufficient                80    80
syntactic_incomplete      80    80

Duplicate trajectory rows: 0
Missing trajectories: 0
Unexpected trajectories: 0

Empty reports: 0
Failed trajectories: 0

PHASE26D EVALUATION GENERATION AUDIT: PASS


In [ ]:
from pathlib import Path

print("Claim-evaluation related files:\n")

for path in sorted(
    Path("src").rglob("*.py")
):
    name = path.name.lower()

    if any(
        term in name
        for term in [
            "claim",
            "evaluate",
            "radgraph",
            "chexpert",
        ]
    ):
        print(path)

print("\nEvaluation-related notebooks/scripts:\n")

for path in sorted(
    Path("notebooks").rglob("*")
):
    if path.is_file():
        print(path)

Claim-evaluation related files:

src/evaluation/claim_evaluator.py
src/evaluation/claim_extractor.py
src/evaluation/claim_patterns.py
src/evaluation/evaluate_abstention_metrics.py
src/evaluation/evaluate_analyzer.py
src/evaluation/evaluate_analyzer_phase12b.py
src/evaluation/evaluate_analyzer_phase12c.py
src/evaluation/evaluate_chexpert_f1.py
src/evaluation/evaluate_gated_generation.py
src/evaluation/evaluate_phase11_paired.py
src/evaluation/evaluate_phase13a_gate.py
src/evaluation/evaluate_phase13b_gated_reports.py
src/evaluation/evaluate_radgraph_f1.py
src/evaluation/evaluate_reference_factuality.py
src/evaluation/evaluate_reliability_gate.py
src/evaluation/evaluate_risk_coverage.py
src/evaluation/evaluate_unsupported_claims.py
src/evaluation/run_phase19_claim_evaluation.py

Evaluation-related notebooks/scripts:



In [ ]:
from pathlib import Path

path = Path("src/evaluation/run_phase19_claim_evaluation.py")

print(path.read_text())

"""Phase 19 paired claim evaluation for the Phase 17/18 576-record cohort.

Uses the established claim-pattern extractor and conservative polarity labels.
This is limited-ontology claim comparison; it does not run RadGraph-F1,
CheXpert-F1, or statistical comparison.
"""

from __future__ import annotations

import hashlib
import sys
from pathlib import Path
from typing import Any

import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.build_reference_candidates import extract_reference_candidates
from src.evaluation.claim_evaluator import load_ontology
from src.evaluation.claim_extractor import extract_generated_claim_candidates
from src.evaluation.claim_patterns import REFERENCE_PATTERNS
from src.evaluation.evaluate_reference_factuality import build_factuality_table

BENCHMARK_PATH = ROOT / "results/tables/phase16_perturbations.csv"
ELIGIBILITY_PATH = ROOT / "results/tables/phase17_vlm_inferen

In [ ]:
path = Path("src/evaluation/claim_evaluator.py")

print(path.read_text())

﻿from pathlib import Path

import pandas as pd
import yaml


CLAIMS_PATH = Path("results/tables/extracted_claims.csv")
CONFLICT_PATH = Path("configs/conflict_cases.yaml")
ONTOLOGY_PATH = Path("configs/claim_ontology.yaml")
OUTPUT_PATH = Path("results/tables/conflict_claim_evaluation.csv")


VALID_POLARITIES = {
    "AFFIRMED",
    "NEGATED",
    "MENTIONED_UNCLEAR",
}


def load_approved_conflicts() -> list[dict]:
    with CONFLICT_PATH.open("r", encoding="utf-8") as file:
        data = yaml.safe_load(file)

    conflicts = data.get("conflicts")

    if not isinstance(conflicts, dict):
        raise ValueError("Expected 'conflicts' to be a mapping.")

    approved_cases = []

    for sample_id, case in conflicts.items():
        if not case.get("approved", False):
            continue

        target = str(case.get("target", "")).strip()

        if not target:
            raise ValueError(
                f"Approved case {sample_id} has an empty target."
            )

        approv

In [ ]:
from pathlib import Path

for filename in [
    "src/evaluation/claim_extractor.py",
    "src/evaluation/evaluate_reference_factuality.py",
    "src/evaluation/build_reference_candidates.py",
]:
    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)

    path = Path(filename)

    print(path.read_text())


src/evaluation/claim_extractor.py
"""
Extract generated-report claim candidates.

This module extracts clinical-finding candidates from baseline
generated reports using the same REFERENCE_PATTERNS vocabulary
and polarity rules as the reference candidate extractor.

It writes:

    results/tables/generated_claim_candidates.csv

It does NOT overwrite:

    results/tables/extracted_claims.csv

Conflict-target evaluation remains in claim_evaluator.py.
That Phase 6 evaluator is condition-blind: it matches claims by
(sample_id, finding) and ignores condition. That behavior is
intentionally left unchanged. This extractor is condition-aware
and keys generated claims by (sample_id, condition, finding).
"""

from pathlib import Path
import sys

import pandas as pd

ROOT = Path(__file__).resolve().parents[2]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.claim_patterns import (
    extract_raw_claim_records,
    strip_generated_report_body,
)


GENERATION_P

In [ ]:
# ============================================================
# Phase26D Evaluation Claim Analysis
#
# Development-selected policy:
#   baseline  = lambda 0.00
#   selected  = lambda 0.25
#
# Evaluation cohort:
#   80 studies × 6 conditions = 480 records
#   2 lambdas = 960 generated reports
#
# Uses the established:
#   - reference candidate extractor
#   - generated claim extractor
#   - reference-grounded factuality evaluator
#
# Does NOT modify the existing evaluation source files.
# ============================================================

from pathlib import Path
import pandas as pd

from src.evaluation.build_reference_candidates import (
    extract_reference_candidates,
)

from src.evaluation.claim_extractor import (
    extract_generated_claim_candidates,
)

from src.evaluation.evaluate_reference_factuality import (
    build_factuality_table,
)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

GENERATION_PATH = Path(
    "results/maira2_contrastive/phase26d_evaluation/"
    "phase26d_evaluation_generation.csv"
)

BENCHMARK_PATH = Path(
    "results/tables/phase16_perturbations.csv"
)

OUTPUT_DIR = Path(
    "results/tables/phase26d_claim_evaluation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Load generated evaluation trajectories
# ------------------------------------------------------------

generation = pd.read_csv(
    GENERATION_PATH
)

print("Generation rows:", len(generation))

assert len(generation) == 960

assert (
    generation["status"]
    .astype(str)
    .eq("success")
    .all()
)

assert (
    generation["lambda"]
    .value_counts()
    .to_dict()
    == {
        0.0: 480,
        0.25: 480,
    }
)


# ------------------------------------------------------------
# Load benchmark
# ------------------------------------------------------------

benchmark = pd.read_csv(
    BENCHMARK_PATH
)

benchmark["uid"] = (
    benchmark["uid"]
    .astype(int)
)


# ------------------------------------------------------------
# Build stable reference table
#
# Findings/impression are identical across the six
# perturbation conditions for a study.
#
# Therefore use one row per study.
# ------------------------------------------------------------

reference_source = (
    benchmark[
        [
            "sample_id",
            "uid",
            "findings",
            "impression",
        ]
    ]
    .drop_duplicates(
        subset=["sample_id", "uid"]
    )
    .reset_index(drop=True)
)

print(
    "Reference studies:",
    len(reference_source)
)

assert len(reference_source) == 100


# ------------------------------------------------------------
# Restrict reference studies to the 80-study evaluation split
# ------------------------------------------------------------

evaluation_uids = set(
    generation["uid"]
    .astype(int)
)

reference_source = (
    reference_source[
        reference_source["uid"]
        .isin(evaluation_uids)
    ]
    .copy()
)

print(
    "Evaluation reference studies:",
    len(reference_source)
)

assert len(reference_source) == 80


# ------------------------------------------------------------
# Extract reference claim candidates
# ------------------------------------------------------------

reference_claims = (
    extract_reference_candidates(
        reference_source[
            [
                "sample_id",
                "findings",
                "impression",
            ]
        ]
    )
)

print(
    "Reference claim candidates:",
    len(reference_claims)
)


# ------------------------------------------------------------
# Prepare generated reports for the established extractor
# ------------------------------------------------------------

generation_for_extractor = (
    generation[
        [
            "sample_id",
            "condition",
            "lambda",
            "report",
        ]
    ]
    .rename(
        columns={
            "report": "generated_report"
        }
    )
)


# ------------------------------------------------------------
# Extract generated claim candidates
#
# The existing extractor requires:
#   sample_id
#   condition
#   generated_report
#
# Lambda is retained separately for pairing.
# ------------------------------------------------------------

generated_claims_by_lambda = []

for lambda_value in [0.0, 0.25]:

    subset = (
        generation_for_extractor[
            generation_for_extractor["lambda"]
            == lambda_value
        ]
        .drop(columns=["lambda"])
        .copy()
    )

    claims = (
        extract_generated_claim_candidates(
            subset
        )
    )

    if not claims.empty:

        claims["lambda"] = (
            float(lambda_value)
        )

    generated_claims_by_lambda.append(
        claims
    )

generated_claims = pd.concat(
    generated_claims_by_lambda,
    ignore_index=True,
)

print(
    "Generated claim candidates:",
    len(generated_claims)
)


# ------------------------------------------------------------
# Build factuality tables separately for each lambda
# ------------------------------------------------------------

factuality_tables = []

for lambda_value in [0.0, 0.25]:

    report_subset = (
        generation[
            generation["lambda"]
            == lambda_value
        ][
            [
                "sample_id",
                "condition",
                "report",
            ]
        ]
        .rename(
            columns={
                "report": "generated_report"
            }
        )
        .copy()
    )

    claim_subset = (
        generated_claims[
            generated_claims["lambda"]
            == lambda_value
        ]
        .drop(
            columns=["lambda"]
        )
        .copy()
    )

    factuality = (
        build_factuality_table(
            report_subset,
            reference_claims,
            claim_subset,
        )
    )

    factuality["lambda"] = (
        float(lambda_value)
    )

    factuality_tables.append(
        factuality
    )


factuality = pd.concat(
    factuality_tables,
    ignore_index=True,
)


# ------------------------------------------------------------
# Add UID
# ------------------------------------------------------------

uid_lookup = (
    generation[
        [
            "sample_id",
            "uid",
        ]
    ]
    .drop_duplicates(
        subset=["sample_id"]
    )
)

factuality = factuality.merge(
    uid_lookup,
    on="sample_id",
    how="left",
    validate="many_to_one",
)


# ------------------------------------------------------------
# Sort and save detailed claim evaluation
# ------------------------------------------------------------

factuality = (
    factuality[
        [
            "sample_id",
            "uid",
            "condition",
            "lambda",
            "finding",
            "generated_polarity",
            "reference_polarity",
            "label",
        ]
    ]
    .sort_values(
        [
            "sample_id",
            "condition",
            "finding",
            "lambda",
        ]
    )
    .reset_index(drop=True)
)


DETAIL_PATH = (
    OUTPUT_DIR
    / "phase26d_evaluation_claims.csv"
)

factuality.to_csv(
    DETAIL_PATH,
    index=False,
)


# ------------------------------------------------------------
# Summary counts
# ------------------------------------------------------------

VALID_LABELS = [
    "supported",
    "omission",
    "extra_negation",
    "ungrounded_affirmation",
    "unsupported_affirmation",
    "contradiction",
    "unclear",
]

UNSUPPORTED_LABELS = {
    "ungrounded_affirmation",
    "unsupported_affirmation",
    "contradiction",
}


summary_rows = []


for lambda_value in [0.0, 0.25]:

    subset = factuality[
        factuality["lambda"]
        == lambda_value
    ]

    label_counts = (
        subset["label"]
        .value_counts()
        .to_dict()
    )

    evaluated = sum(
        int(label_counts.get(label, 0))
        for label in VALID_LABELS
        if label != "omission"
    )

    unsupported = sum(
        int(label_counts.get(label, 0))
        for label in UNSUPPORTED_LABELS
    )

    row = {
        "lambda": lambda_value,
        "comparison_rows": len(subset),
        "supported": int(
            label_counts.get(
                "supported",
                0,
            )
        ),
        "omission": int(
            label_counts.get(
                "omission",
                0,
            )
        ),
        "extra_negation": int(
            label_counts.get(
                "extra_negation",
                0,
            )
        ),
        "ungrounded_affirmation": int(
            label_counts.get(
                "ungrounded_affirmation",
                0,
            )
        ),
        "unsupported_affirmation": int(
            label_counts.get(
                "unsupported_affirmation",
                0,
            )
        ),
        "contradiction": int(
            label_counts.get(
                "contradiction",
                0,
            )
        ),
        "unclear": int(
            label_counts.get(
                "unclear",
                0,
            )
        ),
        "unsupported_claim_count": unsupported,
        "unsupported_claim_rate_denominator": evaluated,
        "unsupported_claim_rate": (
            unsupported / evaluated
            if evaluated
            else 0.0
        ),
    }

    summary_rows.append(
        row
    )


summary = pd.DataFrame(
    summary_rows
)


SUMMARY_PATH = (
    OUTPUT_DIR
    / "phase26d_evaluation_claim_summary.csv"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Condition × lambda summary
# ------------------------------------------------------------

condition_rows = []

for (
    condition,
    lambda_value,
), subset in factuality.groupby(
    [
        "condition",
        "lambda",
    ]
):

    label_counts = (
        subset["label"]
        .value_counts()
        .to_dict()
    )

    evaluated = sum(
        int(label_counts.get(label, 0))
        for label in VALID_LABELS
        if label != "omission"
    )

    unsupported = sum(
        int(label_counts.get(label, 0))
        for label in UNSUPPORTED_LABELS
    )

    condition_rows.append(
        {
            "condition": condition,
            "lambda": lambda_value,
            "comparison_rows": len(
                subset
            ),
            "supported": int(
                label_counts.get(
                    "supported",
                    0,
                )
            ),
            "omission": int(
                label_counts.get(
                    "omission",
                    0,
                )
            ),
            "extra_negation": int(
                label_counts.get(
                    "extra_negation",
                    0,
                )
            ),
            "ungrounded_affirmation": int(
                label_counts.get(
                    "ungrounded_affirmation",
                    0,
                )
            ),
            "unsupported_affirmation": int(
                label_counts.get(
                    "unsupported_affirmation",
                    0,
                )
            ),
            "contradiction": int(
                label_counts.get(
                    "contradiction",
                    0,
                )
            ),
            "unclear": int(
                label_counts.get(
                    "unclear",
                    0,
                )
            ),
            "unsupported_claim_count": unsupported,
            "unsupported_claim_rate_denominator": evaluated,
            "unsupported_claim_rate": (
                unsupported / evaluated
                if evaluated
                else 0.0
            ),
        }
    )


condition_summary = (
    pd.DataFrame(
        condition_rows
    )
    .sort_values(
        [
            "condition",
            "lambda",
        ]
    )
    .reset_index(drop=True)
)


CONDITION_PATH = (
    OUTPUT_DIR
    / "phase26d_evaluation_claim_summary_by_condition.csv"
)

condition_summary.to_csv(
    CONDITION_PATH,
    index=False,
)


# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE26D CLAIM EVALUATION")
print("=" * 70)

print(
    f"Evaluation reports: {len(generation)}"
)

print(
    f"Reference studies: {len(reference_source)}"
)

print(
    f"Reference candidates: {len(reference_claims)}"
)

print(
    f"Generated claim candidates: "
    f"{len(generated_claims)}"
)

print(
    f"Factuality comparison rows: "
    f"{len(factuality)}"
)

print("\nOverall summary:")
print(
    summary.to_string(
        index=False
    )
)

print("\nCondition summary:")
print(
    condition_summary.to_string(
        index=False
    )
)

print("\nOutputs:")
print(
    DETAIL_PATH
)
print(
    SUMMARY_PATH
)
print(
    CONDITION_PATH
)

Generation rows: 960
Reference studies: 100
Evaluation reference studies: 80
Reference claim candidates: 265
Generated claim candidates: 1845

PHASE26D CLAIM EVALUATION
Evaluation reports: 960
Reference studies: 80
Reference candidates: 265
Generated claim candidates: 1845
Factuality comparison rows: 3864

Overall summary:
 lambda  comparison_rows  supported  omission  extra_negation  ungrounded_affirmation  unsupported_affirmation  contradiction  unclear  unsupported_claim_count  unsupported_claim_rate_denominator  unsupported_claim_rate
   0.00             1940        471      1005             269                      91                       35             45       24                      171                                 935                0.182888
   0.25             1924        479       993             271                      77                       38             53       13                      168                                 931                0.180451

Condition summ

In [ ]:
# ============================================================
# Phase26D Paired Statistical Evaluation
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import (
    wilcoxon,
    binomtest,
)


DETAIL_PATH = (
    "results/tables/phase26d_claim_evaluation/"
    "phase26d_evaluation_claims.csv"
)

detail = pd.read_csv(DETAIL_PATH)

UNSUPPORTED_LABELS = {
    "ungrounded_affirmation",
    "unsupported_affirmation",
    "contradiction",
}

# ------------------------------------------------------------
# Unsupported claims per record
# ------------------------------------------------------------

detail["is_unsupported"] = (
    detail["label"]
    .isin(UNSUPPORTED_LABELS)
)


record_rows = []

for (
    sample_id,
    uid,
    condition,
    lambda_value,
), group in detail.groupby(
    [
        "sample_id",
        "uid",
        "condition",
        "lambda",
    ],
    sort=True,
):

    unsupported_count = int(
        group["is_unsupported"].sum()
    )

    evaluated_count = int(
        (
            group["label"]
            != "omission"
        ).sum()
    )

    unsupported_rate = (
        unsupported_count / evaluated_count
        if evaluated_count
        else 0.0
    )

    record_rows.append(
        {
            "sample_id": sample_id,
            "uid": int(uid),
            "condition": condition,
            "lambda": float(lambda_value),
            "unsupported_count": unsupported_count,
            "evaluated_count": evaluated_count,
            "unsupported_rate": unsupported_rate,
            "has_unsupported": (
                unsupported_count > 0
            ),
        }
    )


record_df = pd.DataFrame(
    record_rows
)

# ------------------------------------------------------------
# Pivot baseline and selected policy
# ------------------------------------------------------------

baseline = (
    record_df[
        record_df["lambda"] == 0.0
    ]
    .rename(
        columns={
            "unsupported_count":
                "baseline_unsupported_count",
            "evaluated_count":
                "baseline_evaluated_count",
            "unsupported_rate":
                "baseline_unsupported_rate",
            "has_unsupported":
                "baseline_has_unsupported",
        }
    )
    .drop(columns=["lambda"])
)

selected = (
    record_df[
        record_df["lambda"] == 0.25
    ]
    .rename(
        columns={
            "unsupported_count":
                "selected_unsupported_count",
            "evaluated_count":
                "selected_evaluated_count",
            "unsupported_rate":
                "selected_unsupported_rate",
            "has_unsupported":
                "selected_has_unsupported",
        }
    )
    .drop(columns=["lambda"])
)

paired = baseline.merge(
    selected,
    on=[
        "sample_id",
        "uid",
        "condition",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(paired) == 480

# ------------------------------------------------------------
# Record-level unsupported indicator
# ------------------------------------------------------------

baseline_positive = int(
    paired[
        "baseline_has_unsupported"
    ].sum()
)

selected_positive = int(
    paired[
        "selected_has_unsupported"
    ].sum()
)

baseline_only = int(
    (
        paired["baseline_has_unsupported"]
        &
        ~paired["selected_has_unsupported"]
    ).sum()
)

selected_only = int(
    (
        ~paired["baseline_has_unsupported"]
        &
        paired["selected_has_unsupported"]
    ).sum()
)

both_positive = int(
    (
        paired["baseline_has_unsupported"]
        &
        paired["selected_has_unsupported"]
    ).sum()
)

neither_positive = int(
    (
        ~paired["baseline_has_unsupported"]
        &
        ~paired["selected_has_unsupported"]
    ).sum()
)

# ------------------------------------------------------------
# Exact McNemar test
#
# Conditional on discordant pairs:
# baseline-only vs selected-only
# ------------------------------------------------------------

discordant = (
    baseline_only
    + selected_only
)

if discordant > 0:

    mcnemar = binomtest(
        k=min(
            baseline_only,
            selected_only,
        ),
        n=discordant,
        p=0.5,
        alternative="two-sided",
    )

    mcnemar_p = float(
        mcnemar.pvalue
    )

else:

    mcnemar_p = 1.0


# ------------------------------------------------------------
# Wilcoxon signed-rank test
#
# Per-record unsupported claim rate
# ------------------------------------------------------------

paired["delta_rate"] = (
    paired["selected_unsupported_rate"]
    -
    paired["baseline_unsupported_rate"]
)

nonzero = paired[
    paired["delta_rate"] != 0
]

if len(nonzero) > 0:

    wilcox = wilcoxon(
        nonzero[
            "selected_unsupported_rate"
        ],
        nonzero[
            "baseline_unsupported_rate"
        ],
        alternative="two-sided",
        zero_method="wilcox",
        method="auto",
    )

    wilcoxon_stat = float(
        wilcox.statistic
    )

    wilcoxon_p = float(
        wilcox.pvalue
    )

else:

    wilcoxon_stat = 0.0
    wilcoxon_p = 1.0


# ------------------------------------------------------------
# Rank-biserial effect size
# ------------------------------------------------------------

positive_deltas = int(
    (
        nonzero["delta_rate"]
        > 0
    ).sum()
)

negative_deltas = int(
    (
        nonzero["delta_rate"]
        < 0
    ).sum()
)

if len(nonzero) > 0:

    rank_biserial = (
        negative_deltas
        -
        positive_deltas
    ) / len(nonzero)

else:

    rank_biserial = 0.0


# ------------------------------------------------------------
# Pooled claim-level rates
# ------------------------------------------------------------

pooled = (
    detail.groupby(
        "lambda"
    )
    .agg(
        unsupported_claims=(
            "is_unsupported",
            "sum",
        ),
        evaluated_claims=(
            "is_unsupported",
            "count",
        ),
    )
    .reset_index()
)

# Recompute denominator excluding omissions
pooled_rows = []

for lambda_value, group in detail.groupby(
    "lambda"
):

    unsupported = int(
        group["is_unsupported"].sum()
    )

    evaluated = int(
        (
            group["label"]
            != "omission"
        ).sum()
    )

    pooled_rows.append(
        {
            "lambda": float(lambda_value),
            "unsupported_claims": unsupported,
            "evaluated_claims": evaluated,
            "unsupported_rate": (
                unsupported / evaluated
                if evaluated
                else 0.0
            ),
        }
    )

pooled = pd.DataFrame(
    pooled_rows
)

baseline_rate = float(
    pooled.loc[
        pooled["lambda"] == 0.0,
        "unsupported_rate",
    ].iloc[0]
)

selected_rate = float(
    pooled.loc[
        pooled["lambda"] == 0.25,
        "unsupported_rate",
    ].iloc[0]
)

absolute_change = (
    selected_rate
    -
    baseline_rate
)

relative_reduction = (
    (baseline_rate - selected_rate)
    / baseline_rate
    if baseline_rate
    else 0.0
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 70)
print("PHASE26D PAIRED STATISTICAL EVALUATION")
print("=" * 70)

print(
    f"\nPaired evaluation records: {len(paired)}"
)

print("\nRecord-level unsupported status:")
print(
    f"Baseline positive: {baseline_positive}"
)

print(
    f"λ=0.25 positive: {selected_positive}"
)

print(
    f"Baseline-only: {baseline_only}"
)

print(
    f"λ=0.25-only: {selected_only}"
)

print(
    f"Both positive: {both_positive}"
)

print(
    f"Neither positive: {neither_positive}"
)

print(
    f"Discordant pairs: {discordant}"
)

print(
    f"\nMcNemar exact p-value: "
    f"{mcnemar_p:.6g}"
)

print("\nPer-record unsupported-rate comparison:")

print(
    f"Non-zero paired differences: "
    f"{len(nonzero)}"
)

print(
    f"Wilcoxon statistic: "
    f"{wilcoxon_stat:.6g}"
)

print(
    f"Wilcoxon p-value: "
    f"{wilcoxon_p:.6g}"
)

print(
    f"Rank-biserial effect size: "
    f"{rank_biserial:.6f}"
)

print("\nPooled claim-level risk:")

print(
    pooled.to_string(
        index=False
    )
)

print(
    f"\nAbsolute rate change: "
    f"{absolute_change:.6f}"
)

print(
    f"Relative reduction: "
    f"{relative_reduction:.6%}"
)


# ------------------------------------------------------------
# Save paired record table
# ------------------------------------------------------------

PAIRED_PATH = (
    "results/tables/phase26d_claim_evaluation/"
    "phase26d_evaluation_paired_records.csv"
)

paired.to_csv(
    PAIRED_PATH,
    index=False,
)

print(
    f"\nPaired record output: {PAIRED_PATH}"
)

PHASE26D PAIRED STATISTICAL EVALUATION

Paired evaluation records: 480

Record-level unsupported status:
Baseline positive: 126
λ=0.25 positive: 129
Baseline-only: 30
λ=0.25-only: 33
Both positive: 96
Neither positive: 321
Discordant pairs: 63

McNemar exact p-value: 0.801306

Per-record unsupported-rate comparison:
Non-zero paired differences: 102
Wilcoxon statistic: 2328.5
Wilcoxon p-value: 0.318635
Rank-biserial effect size: -0.078431

Pooled claim-level risk:
 lambda  unsupported_claims  evaluated_claims  unsupported_rate
   0.00                 171               935          0.182888
   0.25                 168               931          0.180451

Absolute rate change: -0.002437
Relative reduction: 1.332278%

Paired record output: results/tables/phase26d_claim_evaluation/phase26d_evaluation_paired_records.csv


In [ ]:
from pathlib import Path

for filename in [
    "src/evaluation/evaluate_radgraph_f1.py",
    "src/evaluation/evaluate_chexpert_f1.py",
]:
    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)

    path = Path(filename)
    print(path.read_text())


src/evaluation/evaluate_radgraph_f1.py
"""
Phase 15 Evaluation: RadGraph-F1 Clinical Content / Entity-Relation Evaluation.

Evaluates entity and relation overlap between generated radiology reports
(baseline and Phase 13B gated reports) and reference reports (reference findings + impression).

Uses the official `radgraph` PyPI package (F1RadGraph / DyGIE model).

Outputs:
    results/tables/radgraph_f1_results.csv
    results/tables/radgraph_f1_summary.csv

Scientific Integrity Note:
    RadGraph-F1 measures clinical entity and relation overlap against reference text.
    It is NOT a direct measure of hallucination rate, factuality accuracy, or safety.
"""

from pathlib import Path
import sys
from typing import Dict, List, Tuple, Any

import pandas as pd
import numpy as np

ROOT = Path(__file__).resolve().parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Monkeypatch transformers for radgraph compatibility with transformers >= 5.0
import transformers

if not

In [ ]:
import importlib.util

print(
    "radgraph installed:",
    importlib.util.find_spec("radgraph") is not None
)

radgraph installed: False


In [ ]:
import sys
import torch
import transformers
import importlib.util

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

print("\nRelevant packages:")
for package in [
    "radgraph",
    "torch",
    "transformers",
    "spacy",
    "nltk",
    "numpy",
    "pandas",
]:
    spec = importlib.util.find_spec(package)
    print(
        f"{package:15s}: "
        f"{'installed' if spec else 'NOT INSTALLED'}"
    )

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
Transformers: 4.51.3
CUDA available: True

Relevant packages:
radgraph       : NOT INSTALLED
torch          : installed
transformers   : installed
spacy          : installed
nltk           : installed
numpy          : installed
pandas         : installed


In [ ]:
!pip install radgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.0/588.0 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for radgraph: filename=radgraph-0.1.18-py3-none-any.whl size=812682 sha256=838a264ea8a1b91b274ba475bd2002fcc673b2cfc0c3132388c3ba6707aab75c
  Stored in directory: /root/.cache/pip/wheels/2d/2a/49/a9fd7a99361f4d67e884f047d011ac9a619f6f2575ae3c9851
Successfully built radgraph


In [ ]:
import radgraph
print("RadGraph imported successfully")

RadGraph imported successfully


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from radgraph import F1RadGraph

print("F1RadGraph imported successfully")
print(F1RadGraph)

F1RadGraph imported successfully
<class 'radgraph.radgraph.F1RadGraph'>


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys

# ------------------------------------------------------------
# Phase26D paths
# ------------------------------------------------------------
ROOT = Path("/content/Evidence-State-Radiology")

GEN_PATH = (
    ROOT
    / "results/maira2_contrastive/phase26d_evaluation"
    / "phase26d_evaluation_generation.csv"
)

BENCH_PATH = (
    ROOT
    / "results/tables/phase16_perturbations.csv"
)

OUT_DIR = ROOT / "results/tables/phase26d_metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load generation results
# ------------------------------------------------------------
generation_df = pd.read_csv(GEN_PATH)
benchmark_df = pd.read_csv(BENCH_PATH)

print("Generation shape:", generation_df.shape)
print("Benchmark shape:", benchmark_df.shape)

print("\nGeneration columns:")
print(generation_df.columns.tolist())

print("\nBenchmark columns:")
print(benchmark_df.columns.tolist())

Generation shape: (960, 14)
Benchmark shape: (600, 20)

Generation columns:
['sample_id', 'uid', 'condition', 'evidence_state', 'perturbed_context', 'frontal_image', 'findings', 'impression', 'lambda', 'report', 'num_tokens', 'runtime_seconds', 'status', 'error']

Benchmark columns:
['sample_id', 'uid', 'condition', 'evidence_state', 'original_context', 'perturbed_context', 'frontal_image', 'lateral_image', 'findings', 'impression', 'comparison', 'source_context_uid', 'source_context_sample_id', 'removed_evidence_text', 'evidentiary_incomplete_method', 'conflict_target', 'conflict_method', 'conflict_source_text', 'generation_status', 'generation_failure_reason']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# ------------------------------------------------------------
# Phase26D: Build evaluation reference table
# ------------------------------------------------------------

# Evaluation studies are the studies appearing in the generated
# Phase26D evaluation outputs.
eval_uids = sorted(generation_df["uid"].dropna().unique())

print("Evaluation studies:", len(eval_uids))

# ------------------------------------------------------------
# Reference source
# One stable clinical reference per study.
# Use findings + impression from the frozen benchmark.
# ------------------------------------------------------------

reference_df = (
    benchmark_df[
        benchmark_df["uid"].isin(eval_uids)
    ][
        ["uid", "findings", "impression"]
    ]
    .drop_duplicates(subset=["uid"])
    .copy()
)

# Construct the same reference text for RadGraph as the
# existing Phase20 implementation:
# findings + impression.
reference_df["reference_text"] = (
    reference_df["findings"]
    .fillna("")
    .astype(str)
    .str.strip()
    + "\n"
    + reference_df["impression"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("Reference studies:", len(reference_df))
print(
    "Reference text empty:",
    (reference_df["reference_text"].str.len() == 0).sum()
)

print(
    "Duplicate reference UIDs:",
    reference_df["uid"].duplicated().sum()
)

missing_refs = sorted(
    set(eval_uids) - set(reference_df["uid"])
)

print("Missing evaluation references:", missing_refs)

print("\nReference length statistics:")
print(
    reference_df["reference_text"]
    .str.len()
    .describe()
)

# ------------------------------------------------------------
# Verify every generated record has a reference
# ------------------------------------------------------------

generation_with_ref = generation_df.merge(
    reference_df[["uid", "reference_text"]],
    on="uid",
    how="left",
    validate="many_to_one",
)

print("\nGeneration + reference shape:", generation_with_ref.shape)

print(
    "Generated rows without reference:",
    generation_with_ref["reference_text"].isna().sum()
)

print(
    "Unique generated reports:",
    generation_with_ref["report"].nunique()
)

# ------------------------------------------------------------
# Final structural audit
# ------------------------------------------------------------

expected_per_lambda = 480

print("\nLambda counts:")
print(generation_with_ref["lambda"].value_counts().sort_index())

print("\nCondition counts:")
print(generation_with_ref["condition"].value_counts().sort_index())

print("\nCondition × lambda:")
print(
    generation_with_ref
    .groupby(["condition", "lambda"])
    .size()
    .unstack(fill_value=0)
)

assert len(eval_uids) == 80
assert len(reference_df) == 80
assert generation_with_ref["reference_text"].notna().all()
assert generation_with_ref["lambda"].value_counts().to_dict() == {
    0.00: 480,
    0.25: 480,
}

print("\nREFERENCE AUDIT: PASS")

Evaluation studies: 80
Reference studies: 80
Reference text empty: 0
Duplicate reference UIDs: 0
Missing evaluation references: []

Reference length statistics:
count      80.000000
mean      338.462500
std       183.826552
min        91.000000
25%       215.000000
50%       272.500000
75%       436.000000
max      1065.000000
Name: reference_text, dtype: float64

Generation + reference shape: (960, 15)
Generated rows without reference: 0
Unique generated reports: 622

Lambda counts:
lambda
0.00    480
0.25    480
Name: count, dtype: int64

Condition counts:
condition
conflicting               160
evidentiary_incomplete    160
insufficient              160
irrelevant                160
sufficient                160
syntactic_incomplete      160
Name: count, dtype: int64

Condition × lambda:
lambda                  0.00  0.25
condition                         
conflicting               80    80
evidentiary_incomplete    80    80
insufficient              80    80
irrelevant             

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from radgraph import F1RadGraph
import pandas as pd
import numpy as np
from pathlib import Path
import time

# ------------------------------------------------------------
# Phase26D RadGraph evaluation
# ------------------------------------------------------------

OUT_DIR = ROOT / "results/tables/phase26d_metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Make sure reports are valid strings.
radgraph_input = generation_with_ref[
    [
        "sample_id",
        "uid",
        "condition",
        "lambda",
        "report",
        "reference_text",
    ]
].copy()

radgraph_input["report"] = (
    radgraph_input["report"]
    .fillna("")
    .astype(str)
)

radgraph_input["reference_text"] = (
    radgraph_input["reference_text"]
    .fillna("")
    .astype(str)
)

assert len(radgraph_input) == 960
assert radgraph_input["report"].str.len().gt(0).all()

# ------------------------------------------------------------
# Initialize RadGraph once
# ------------------------------------------------------------

print("Initializing F1RadGraph...")
start_init = time.time()

radgraph_metric = F1RadGraph(
    reward_level="all",
    batch_size=8,
)

print(
    f"RadGraph initialized in "
    f"{time.time() - start_init:.2f} seconds"
)

# ------------------------------------------------------------
# Evaluate each lambda separately
# ------------------------------------------------------------

all_results = []

for lambda_value in sorted(radgraph_input["lambda"].unique()):

    subset = (
        radgraph_input[
            radgraph_input["lambda"] == lambda_value
        ]
        .copy()
        .reset_index(drop=True)
    )

    print(
        f"\nRunning RadGraph for lambda={lambda_value:.2f} "
        f"({len(subset)} records)..."
    )

    references = subset["reference_text"].tolist()
    hypotheses = subset["report"].tolist()

    start = time.time()

    scores = radgraph_metric(
        refs=references,
        hyps=hypotheses,
    )

    elapsed = time.time() - start

    print(
        f"Completed lambda={lambda_value:.2f} "
        f"in {elapsed:.2f} seconds"
    )

    # --------------------------------------------------------
    # Inspect returned structure before assigning
    # --------------------------------------------------------

    print("Returned object type:", type(scores))

    if isinstance(scores, dict):
        print("Returned keys:", scores.keys())

    # --------------------------------------------------------
    # Existing F1RadGraph returns a dictionary containing
    # precision / recall / f1_score lists.
    # --------------------------------------------------------

    precision = scores["precision"]
    recall = scores["recall"]
    f1 = scores["f1_score"]

    assert len(precision) == len(subset)
    assert len(recall) == len(subset)
    assert len(f1) == len(subset)

    subset["radgraph_precision"] = precision
    subset["radgraph_recall"] = recall
    subset["radgraph_f1"] = f1

    all_results.append(subset)

radgraph_results = pd.concat(
    all_results,
    ignore_index=True,
)

# ------------------------------------------------------------
# Structural audit
# ------------------------------------------------------------

print("\nRadGraph result shape:", radgraph_results.shape)

print("\nLambda counts:")
print(
    radgraph_results["lambda"]
    .value_counts()
    .sort_index()
)

print("\nCondition counts:")
print(
    radgraph_results["condition"]
    .value_counts()
    .sort_index()
)

print("\nNaN counts:")
print(
    radgraph_results[
        [
            "radgraph_precision",
            "radgraph_recall",
            "radgraph_f1",
        ]
    ].isna().sum()
)

print("\nOverall RadGraph means:")
print(
    radgraph_results
    .groupby("lambda")[
        [
            "radgraph_precision",
            "radgraph_recall",
            "radgraph_f1",
        ]
    ]
    .mean()
)

print("\nOverall RadGraph medians:")
print(
    radgraph_results
    .groupby("lambda")[
        [
            "radgraph_precision",
            "radgraph_recall",
            "radgraph_f1",
        ]
    ]
    .median()
)

# ------------------------------------------------------------
# Save record-level results
# ------------------------------------------------------------

RADGRAPH_RECORD_PATH = (
    OUT_DIR / "phase26d_evaluation_radgraph.csv"
)

radgraph_results.to_csv(
    RADGRAPH_RECORD_PATH,
    index=False,
)

print(
    "\nSaved:",
    RADGRAPH_RECORD_PATH
)

assert len(radgraph_results) == 960
assert radgraph_results["radgraph_f1"].notna().all()

print("\nRADGRAPH RECORD-LEVEL AUDIT: PASS")

Initializing F1RadGraph...
Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


radgraph-xl.tar.gz:   0%|          | 0.00/416M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:1727: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(
/usr/local/lib/python3.13/dist-packages/radgraph/radgraph.py:105: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=model_dir)


tokenizer_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

RadGraph initialized in 30.22 seconds

Running RadGraph for lambda=0.00 (480 records)...
Completed lambda=0.00 in 28.49 seconds
Returned object type: <class 'tuple'>


TypeError: tuple indices must be integers or slices, not str

In [ ]:
print("Type:", type(scores))
print("Tuple length:", len(scores))

for i, item in enumerate(scores):
    print(f"\nItem {i}:")
    print("  type:", type(item))

    if hasattr(item, "shape"):
        print("  shape:", item.shape)

    if isinstance(item, (list, tuple)):
        print("  length:", len(item))
        print("  first item:", item[0] if len(item) > 0 else None)

    elif isinstance(item, dict):
        print("  keys:", item.keys())

    else:
        print("  value:", item)

Type: <class 'tuple'>
Tuple length: 4

Item 0:
  type: <class 'tuple'>
  length: 3
  first item: 0.1881163061476282

Item 1:
  type: <class 'tuple'>
  length: 3
  first item: [0.0, 0.14814814814814817, 0.15384615384615385, 0.0, 0.06250000000000001, 0.24242424242424243, 0.0, 0.22641509433962262, 0.3846153846153846, 0.18181818181818182, 0.37037037037037035, 0.24999999999999994, 0.14285714285714285, 0.13793103448275862, 0.2962962962962963, 0.14634146341463414, 0.12500000000000003, 0.3125, 0.15384615384615385, 0.13793103448275862, 0.11764705882352941, 0.0, 0.2, 0.1702127659574468, 0.052631578947368425, 0.2564102564102564, 0.0625, 0.1111111111111111, 0.1111111111111111, 0.09523809523809523, 0.14285714285714285, 0.18181818181818182, 0.30769230769230765, 0.05405405405405406, 0.28571428571428575, 0.16216216216216214, 0.0, 0.18181818181818182, 0.058823529411764705, 0.1111111111111111, 0.30303030303030304, 0.2857142857142857, 0.21276595744680848, 0.2380952380952381, 0.30303030303030304, 0.105263

In [ ]:
# Inspect the first per-record RadGraph score only.
# No model inference is performed here.

print("Aggregate score:", scores[0])

print("\nFirst per-record score:")
print(scores[1][0])

print("\nType:")
print(type(scores[1][0]))

print("\nLength:")
print(len(scores[1][0]))

print("\nParsed hypothesis keys:")
print(scores[2][0].keys())

print("\nParsed reference keys:")
print(scores[3][0].keys())

Aggregate score: (np.float64(0.1881163061476282), np.float64(0.1657536190510923), np.float64(0.11823509276005441))

First per-record score:
[0.0, 0.14814814814814817, 0.15384615384615385, 0.0, 0.06250000000000001, 0.24242424242424243, 0.0, 0.22641509433962262, 0.3846153846153846, 0.18181818181818182, 0.37037037037037035, 0.24999999999999994, 0.14285714285714285, 0.13793103448275862, 0.2962962962962963, 0.14634146341463414, 0.12500000000000003, 0.3125, 0.15384615384615385, 0.13793103448275862, 0.11764705882352941, 0.0, 0.2, 0.1702127659574468, 0.052631578947368425, 0.2564102564102564, 0.0625, 0.1111111111111111, 0.1111111111111111, 0.09523809523809523, 0.14285714285714285, 0.18181818181818182, 0.30769230769230765, 0.05405405405405406, 0.28571428571428575, 0.16216216216216214, 0.0, 0.18181818181818182, 0.058823529411764705, 0.1111111111111111, 0.30303030303030304, 0.2857142857142857, 0.21276595744680848, 0.2380952380952381, 0.30303030303030304, 0.10526315789473685, 0.28571428571428575, 0

In [ ]:
# ------------------------------------------------------------
# Preserve already-computed lambda=0 RadGraph result
# ------------------------------------------------------------

scores_lambda0 = scores

assert len(scores_lambda0) == 4
assert len(scores_lambda0[1]) == 3

precision0 = scores_lambda0[1][0]
recall0 = scores_lambda0[1][1]
f10 = scores_lambda0[1][2]

assert len(precision0) == 480
assert len(recall0) == 480
assert len(f10) == 480

print("Lambda=0 result preserved.")
print("Aggregate precision:", scores_lambda0[0][0])
print("Aggregate recall:", scores_lambda0[0][1])
print("Aggregate F1:", scores_lambda0[0][2])

print("\nPer-record lengths:")
print("Precision:", len(precision0))
print("Recall:", len(recall0))
print("F1:", len(f10))

# ------------------------------------------------------------
# Run lambda=0.25
# ------------------------------------------------------------

lambda025_df = (
    radgraph_input[
        radgraph_input["lambda"] == 0.25
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(lambda025_df) == 480

print("\nRunning RadGraph for lambda=0.25...")
start = time.time()

scores_lambda025 = radgraph_metric(
    refs=lambda025_df["reference_text"].tolist(),
    hyps=lambda025_df["report"].tolist(),
)

elapsed = time.time() - start

print(f"Completed in {elapsed:.2f} seconds")

# ------------------------------------------------------------
# Validate lambda=0.25 structure
# ------------------------------------------------------------

assert len(scores_lambda025) == 4
assert len(scores_lambda025[1]) == 3

precision025 = scores_lambda025[1][0]
recall025 = scores_lambda025[1][1]
f1025 = scores_lambda025[1][2]

assert len(precision025) == 480
assert len(recall025) == 480
assert len(f1025) == 480

print("\nLambda=0.25 aggregate:")
print("Precision:", scores_lambda025[0][0])
print("Recall:", scores_lambda025[0][1])
print("F1:", scores_lambda025[0][2])

print("\nPer-record lengths:")
print("Precision:", len(precision025))
print("Recall:", len(recall025))
print("F1:", len(f1025))

print("\nRADGRAPH PHASE26D AUDIT: PASS")

Lambda=0 result preserved.
Aggregate precision: 0.1881163061476282
Aggregate recall: 0.1657536190510923
Aggregate F1: 0.11823509276005441

Per-record lengths:
Precision: 480
Recall: 480
F1: 480

Running RadGraph for lambda=0.25...
Completed in 23.54 seconds

Lambda=0.25 aggregate:
Precision: 0.18586658444799517
Recall: 0.16482982886097705
F1: 0.12046682595491279

Per-record lengths:
Precision: 480
Recall: 480
F1: 480

RADGRAPH PHASE26D AUDIT: PASS


In [ ]:
# ============================================================
# Phase26D RadGraph — record-level results and summaries
# ============================================================

from scipy.stats import wilcoxon

RADGRAPH_OUT = (
    ROOT / "results/tables/phase26d_metrics"
)
RADGRAPH_OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Attach scores to the original generation ordering
# ------------------------------------------------------------

lambda0_results = (
    radgraph_input[
        radgraph_input["lambda"] == 0.00
    ]
    .copy()
    .reset_index(drop=True)
)

lambda025_results = (
    radgraph_input[
        radgraph_input["lambda"] == 0.25
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(lambda0_results) == 480
assert len(lambda025_results) == 480

# Scores correspond to the exact ordering supplied to RadGraph.
lambda0_results["radgraph_precision"] = precision0
lambda0_results["radgraph_recall"] = recall0
lambda0_results["radgraph_f1"] = f10

lambda025_results["radgraph_precision"] = precision025
lambda025_results["radgraph_recall"] = recall025
lambda025_results["radgraph_f1"] = f1025

radgraph_results = pd.concat(
    [lambda0_results, lambda025_results],
    ignore_index=True,
)

# ------------------------------------------------------------
# Record-level output
# ------------------------------------------------------------

record_path = (
    RADGRAPH_OUT
    / "phase26d_evaluation_radgraph.csv"
)

radgraph_results.to_csv(
    record_path,
    index=False,
)

# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------

overall_summary = (
    radgraph_results
    .groupby("lambda")
    .agg(
        n=("radgraph_f1", "size"),
        precision_mean=("radgraph_precision", "mean"),
        precision_median=("radgraph_precision", "median"),
        recall_mean=("radgraph_recall", "mean"),
        recall_median=("radgraph_recall", "median"),
        f1_mean=("radgraph_f1", "mean"),
        f1_median=("radgraph_f1", "median"),
    )
    .reset_index()
)

overall_path = (
    RADGRAPH_OUT
    / "phase26d_evaluation_radgraph_summary.csv"
)

overall_summary.to_csv(
    overall_path,
    index=False,
)

# ------------------------------------------------------------
# Condition-level summary
# ------------------------------------------------------------

condition_summary = (
    radgraph_results
    .groupby(["condition", "lambda"])
    .agg(
        n=("radgraph_f1", "size"),
        precision_mean=("radgraph_precision", "mean"),
        precision_median=("radgraph_precision", "median"),
        recall_mean=("radgraph_recall", "mean"),
        recall_median=("radgraph_recall", "median"),
        f1_mean=("radgraph_f1", "mean"),
        f1_median=("radgraph_f1", "median"),
    )
    .reset_index()
    .sort_values(["condition", "lambda"])
)

condition_path = (
    RADGRAPH_OUT
    / "phase26d_evaluation_radgraph_by_condition.csv"
)

condition_summary.to_csv(
    condition_path,
    index=False,
)

# ------------------------------------------------------------
# Paired record-level analysis
# ------------------------------------------------------------

paired = (
    lambda0_results[
        [
            "sample_id",
            "uid",
            "condition",
            "radgraph_precision",
            "radgraph_recall",
            "radgraph_f1",
        ]
    ]
    .rename(
        columns={
            "radgraph_precision": "precision_lambda0",
            "radgraph_recall": "recall_lambda0",
            "radgraph_f1": "f1_lambda0",
        }
    )
    .merge(
        lambda025_results[
            [
                "sample_id",
                "uid",
                "condition",
                "radgraph_precision",
                "radgraph_recall",
                "radgraph_f1",
            ]
        ].rename(
            columns={
                "radgraph_precision": "precision_lambda025",
                "radgraph_recall": "recall_lambda025",
                "radgraph_f1": "f1_lambda025",
            }
        ),
        on=["sample_id", "uid", "condition"],
        how="inner",
        validate="one_to_one",
    )
)

assert len(paired) == 480

paired["precision_delta"] = (
    paired["precision_lambda025"]
    - paired["precision_lambda0"]
)

paired["recall_delta"] = (
    paired["recall_lambda025"]
    - paired["recall_lambda0"]
)

paired["f1_delta"] = (
    paired["f1_lambda025"]
    - paired["f1_lambda0"]
)

# ------------------------------------------------------------
# Wilcoxon signed-rank test on paired F1
# ------------------------------------------------------------

nonzero_f1 = paired.loc[
    paired["f1_delta"] != 0,
    "f1_delta"
]

wilcoxon_result = wilcoxon(
    paired["f1_lambda025"],
    paired["f1_lambda0"],
    alternative="two-sided",
    zero_method="wilcox",
)

stat = float(wilcoxon_result.statistic)
p_value = float(wilcoxon_result.pvalue)

# Rank-biserial correlation
n_nonzero = len(nonzero_f1)

if n_nonzero > 0:
    positive_rank_sum = (
        nonzero_f1[nonzero_f1 > 0]
        .rank(method="average")
        .sum()
    )
    negative_rank_sum = (
        (-nonzero_f1[nonzero_f1 < 0])
        .rank(method="average")
        .sum()
    )

    rank_biserial = (
        positive_rank_sum - negative_rank_sum
    ) / (
        positive_rank_sum + negative_rank_sum
    )
else:
    rank_biserial = np.nan

paired_path = (
    RADGRAPH_OUT
    / "phase26d_evaluation_radgraph_paired.csv"
)

paired.to_csv(
    paired_path,
    index=False,
)

stats_df = pd.DataFrame(
    [
        {
            "metric": "radgraph_f1",
            "n_pairs": len(paired),
            "n_nonzero_differences": n_nonzero,
            "lambda0_mean": paired["f1_lambda0"].mean(),
            "lambda025_mean": paired["f1_lambda025"].mean(),
            "mean_delta_lambda025_minus_lambda0":
                paired["f1_delta"].mean(),
            "median_delta":
                paired["f1_delta"].median(),
            "wilcoxon_statistic": stat,
            "wilcoxon_p_value": p_value,
            "rank_biserial": rank_biserial,
        }
    ]
)

stats_path = (
    RADGRAPH_OUT
    / "phase26d_evaluation_radgraph_paired_stats.csv"
)

stats_df.to_csv(
    stats_path,
    index=False,
)

# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

assert len(radgraph_results) == 960
assert radgraph_results["radgraph_f1"].notna().all()
assert len(overall_summary) == 2
assert len(condition_summary) == 12
assert len(paired) == 480

print("RECORD RESULTS:", record_path)
print("OVERALL SUMMARY:", overall_path)
print("CONDITION SUMMARY:", condition_path)
print("PAIRED RESULTS:", paired_path)
print("PAIRED STATS:", stats_path)

print("\nOVERALL:")
display(overall_summary)

print("\nCONDITION:")
display(condition_summary)

print("\nPAIRED RADGRAPH-F1:")
display(stats_df)

print("\nRADGRAPH PHASE26D FINAL AUDIT: PASS")

RECORD RESULTS: /content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_radgraph.csv
OVERALL SUMMARY: /content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_radgraph_summary.csv
CONDITION SUMMARY: /content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_radgraph_by_condition.csv
PAIRED RESULTS: /content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_radgraph_paired.csv
PAIRED STATS: /content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_radgraph_paired_stats.csv

OVERALL:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,lambda,n,precision_mean,precision_median,recall_mean,recall_median,f1_mean,f1_median
0,0.00,480,0.188116,0.181818,0.165754,0.153846,0.118235,0.100847
1,0.25,480,0.185867,0.177778,0.164830,0.163333,0.120467,0.102130



CONDITION:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,condition,lambda,n,precision_mean,precision_median,recall_mean,recall_median,f1_mean,f1_median
0,conflicting,0.00,80,0.170817,0.160028,0.154465,0.149074,0.097833,0.076923
1,conflicting,0.25,80,0.161447,0.155870,0.143300,0.144599,0.097971,0.084220
2,evidentiary_incomplete,0.00,80,0.251074,0.271499,0.218779,0.219219,0.158154,0.142857
3,evidentiary_incomplete,0.25,80,0.225557,0.250000,0.199290,0.193277,0.156858,0.155870
4,insufficient,0.00,80,0.056178,0.000000,0.045467,0.000000,0.030178,0.000000
5,insufficient,0.25,80,0.056178,0.000000,0.045467,0.000000,0.030178,0.000000
6,irrelevant,0.00,80,0.226850,0.211310,0.205611,0.175192,0.156038,0.130496
7,irrelevant,0.25,80,0.212722,0.203448,0.191661,0.184659,0.142981,0.151923
8,sufficient,0.00,80,0.230059,0.244289,0.200675,0.200000,0.147927,0.142857
9,sufficient,0.25,80,0.243202,0.243163,0.216575,0.209430,0.160245,0.148171



PAIRED RADGRAPH-F1:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,metric,n_pairs,n_nonzero_differences,lambda0_mean,lambda025_mean,mean_delta_lambda025_minus_lambda0,median_delta,wilcoxon_statistic,wilcoxon_p_value,rank_biserial
0,radgraph_f1,480,277,0.118235,0.120467,0.002232,0.0,17968.5,0.336329,0.150224



RADGRAPH PHASE26D FINAL AUDIT: PASS


In [ ]:
import sys
from pathlib import Path

ROOT = Path("/content/Evidence-State-Radiology")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.evaluate_chexpert_f1 import (
    CHEXPERT_CATEGORIES,
    extract_chexpert_vector,
    compute_chexpert_f1,
)

print("CheXpert implementation imported successfully.")
print("Number of categories:", len(CHEXPERT_CATEGORIES))
print("\nCategories:")
for i, category in enumerate(CHEXPERT_CATEGORIES, start=1):
    print(f"{i:2d}. {category}")

CheXpert implementation imported successfully.
Number of categories: 14

Categories:
 1. Enlarged Cardiomediastinum
 2. Cardiomegaly
 3. Lung Opacity
 4. Lung Lesion
 5. Edema
 6. Consolidation
 7. Pneumonia
 8. Atelectasis
 9. Pneumothorax
10. Pleural Effusion
11. Pleural Other
12. Fracture
13. Support Devices
14. No Finding


In [ ]:
import inspect

print("extract_chexpert_vector signature:")
print(inspect.signature(extract_chexpert_vector))

print("\ncompute_chexpert_f1 signature:")
print(inspect.signature(compute_chexpert_f1))

print("\ncompute_chexpert_f1 source:")
print(inspect.getsource(compute_chexpert_f1))

extract_chexpert_vector signature:
(text: str) -> Dict[str, Any]

compute_chexpert_f1 signature:
(ref_vector: Dict[str, Any], cand_vector: Dict[str, Any]) -> float

compute_chexpert_f1 source:
def compute_chexpert_f1(ref_vector: Dict[str, Any], cand_vector: Dict[str, Any]) -> float:
    """
    Compute micro CheXpert-F1 between reference vector and candidate vector.
    
    Positive targets are conditions where ref_vector == 1.
    """
    tp = 0
    fp = 0
    fn = 0

    for cat in CHEXPERT_CATEGORIES:
        r_val = ref_vector[cat]
        c_val = cand_vector[cat]

        r_pos = (r_val == 1)
        c_pos = (c_val == 1)

        if r_pos and c_pos:
            tp += 1
        elif not r_pos and c_pos:
            fp += 1
        elif r_pos and not c_pos:
            fn += 1

    if tp == 0 and fp == 0 and fn == 0:
        return 1.0  # Perfect agreement on absence of positive findings

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + 

In [ ]:
# ============================================================
# Phase26D — Custom 14-category CheXpert-F1 evaluation
# Corrected for the "lambda" reserved-keyword issue
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

CHEXPERT_OUT = (
    ROOT / "results/tables/phase26d_metrics"
)
CHEXPERT_OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Prepare input
# ------------------------------------------------------------

chexpert_input = generation_with_ref[
    [
        "sample_id",
        "uid",
        "condition",
        "lambda",
        "report",
        "reference_text",
    ]
].copy()

chexpert_input["report"] = (
    chexpert_input["report"]
    .fillna("")
    .astype(str)
)

chexpert_input["reference_text"] = (
    chexpert_input["reference_text"]
    .fillna("")
    .astype(str)
)

assert len(chexpert_input) == 960
assert chexpert_input["report"].str.len().gt(0).all()
assert chexpert_input["reference_text"].str.len().gt(0).all()

print("Input records:", len(chexpert_input))

# ------------------------------------------------------------
# Extract reference vectors once per study
# ------------------------------------------------------------

reference_unique = (
    chexpert_input[
        ["uid", "reference_text"]
    ]
    .drop_duplicates(subset=["uid"])
    .reset_index(drop=True)
)

assert len(reference_unique) == 80

reference_vectors = {}

for _, ref_row in reference_unique.iterrows():
    reference_vectors[ref_row["uid"]] = (
        extract_chexpert_vector(
            ref_row["reference_text"]
        )
    )

# Audit reference vectors
assert len(reference_vectors) == 80

for uid, vector in reference_vectors.items():
    assert set(vector.keys()) == set(CHEXPERT_CATEGORIES)

print("Reference vector audit: PASS")

# ------------------------------------------------------------
# Extract candidate vectors + calculate F1
#
# Use iterrows() here so "lambda" remains an ordinary
# dictionary-style column access and cannot conflict with
# Python's reserved keyword.
# ------------------------------------------------------------

results = []

for _, row in chexpert_input.iterrows():

    ref_vector = reference_vectors[row["uid"]]

    cand_vector = extract_chexpert_vector(
        row["report"]
    )

    score = compute_chexpert_f1(
        ref_vector,
        cand_vector,
    )

    results.append(
        {
            "sample_id": row["sample_id"],
            "uid": row["uid"],
            "condition": row["condition"],
            "lambda": row["lambda"],
            "chexpert_f1": score,
        }
    )

chexpert_results = pd.DataFrame(results)

# ------------------------------------------------------------
# Structural audit
# ------------------------------------------------------------

assert len(chexpert_results) == 960
assert chexpert_results["chexpert_f1"].notna().all()

lambda_counts = (
    chexpert_results["lambda"]
    .value_counts()
    .sort_index()
    .to_dict()
)

print("\nLambda counts:")
print(lambda_counts)

assert lambda_counts == {
    0.00: 480,
    0.25: 480,
}

condition_counts = (
    chexpert_results["condition"]
    .value_counts()
)

print("\nCondition counts:")
print(condition_counts)

assert condition_counts.eq(160).all()

assert chexpert_results["chexpert_f1"].between(
    0.0,
    1.0
).all()

print("\nCHEXPERT RECORD AUDIT: PASS")

# ------------------------------------------------------------
# Save record-level results
# ------------------------------------------------------------

record_path = (
    CHEXPERT_OUT
    / "phase26d_evaluation_chexpert.csv"
)

chexpert_results.to_csv(
    record_path,
    index=False,
)

# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------

overall_summary = (
    chexpert_results
    .groupby("lambda")
    .agg(
        n=("chexpert_f1", "size"),
        f1_mean=("chexpert_f1", "mean"),
        f1_median=("chexpert_f1", "median"),
    )
    .reset_index()
)

overall_path = (
    CHEXPERT_OUT
    / "phase26d_evaluation_chexpert_summary.csv"
)

overall_summary.to_csv(
    overall_path,
    index=False,
)

# ------------------------------------------------------------
# Condition-level summary
# ------------------------------------------------------------

condition_summary = (
    chexpert_results
    .groupby(["condition", "lambda"])
    .agg(
        n=("chexpert_f1", "size"),
        f1_mean=("chexpert_f1", "mean"),
        f1_median=("chexpert_f1", "median"),
    )
    .reset_index()
    .sort_values(["condition", "lambda"])
)

condition_path = (
    CHEXPERT_OUT
    / "phase26d_evaluation_chexpert_by_condition.csv"
)

condition_summary.to_csv(
    condition_path,
    index=False,
)

# ------------------------------------------------------------
# Paired lambda=0 vs lambda=.25
# ------------------------------------------------------------

lambda0 = (
    chexpert_results[
        chexpert_results["lambda"] == 0.00
    ][
        [
            "sample_id",
            "uid",
            "condition",
            "chexpert_f1",
        ]
    ]
    .rename(
        columns={
            "chexpert_f1": "f1_lambda0"
        }
    )
)

lambda025 = (
    chexpert_results[
        chexpert_results["lambda"] == 0.25
    ][
        [
            "sample_id",
            "uid",
            "condition",
            "chexpert_f1",
        ]
    ]
    .rename(
        columns={
            "chexpert_f1": "f1_lambda025"
        }
    )
)

paired = lambda0.merge(
    lambda025,
    on=[
        "sample_id",
        "uid",
        "condition",
    ],
    how="inner",
    validate="one_to_one",
)

assert len(paired) == 480

paired["f1_delta"] = (
    paired["f1_lambda025"]
    - paired["f1_lambda0"]
)

# ------------------------------------------------------------
# Wilcoxon signed-rank test
# ------------------------------------------------------------

wilcoxon_result = wilcoxon(
    paired["f1_lambda025"],
    paired["f1_lambda0"],
    alternative="two-sided",
    zero_method="wilcox",
)

stat = float(wilcoxon_result.statistic)
p_value = float(wilcoxon_result.pvalue)

nonzero = paired.loc[
    paired["f1_delta"] != 0,
    "f1_delta"
]

n_nonzero = len(nonzero)

# ------------------------------------------------------------
# Rank-biserial correlation
# ------------------------------------------------------------

if n_nonzero > 0:

    positive_rank_sum = (
        nonzero[nonzero > 0]
        .rank(method="average")
        .sum()
    )

    negative_rank_sum = (
        (-nonzero[nonzero < 0])
        .rank(method="average")
        .sum()
    )

    rank_biserial = (
        positive_rank_sum - negative_rank_sum
    ) / (
        positive_rank_sum + negative_rank_sum
    )

else:
    rank_biserial = np.nan

# ------------------------------------------------------------
# Save paired results
# ------------------------------------------------------------

paired_path = (
    CHEXPERT_OUT
    / "phase26d_evaluation_chexpert_paired.csv"
)

paired.to_csv(
    paired_path,
    index=False,
)

stats_df = pd.DataFrame(
    [
        {
            "metric": "chexpert_f1",
            "n_pairs": len(paired),
            "n_nonzero_differences": n_nonzero,
            "lambda0_mean": paired["f1_lambda0"].mean(),
            "lambda025_mean": paired["f1_lambda025"].mean(),
            "mean_delta_lambda025_minus_lambda0":
                paired["f1_delta"].mean(),
            "median_delta":
                paired["f1_delta"].median(),
            "wilcoxon_statistic": stat,
            "wilcoxon_p_value": p_value,
            "rank_biserial": rank_biserial,
        }
    ]
)

stats_path = (
    CHEXPERT_OUT
    / "phase26d_evaluation_chexpert_paired_stats.csv"
)

stats_df.to_csv(
    stats_path,
    index=False,
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nRECORD RESULTS:")
print(record_path)

print("\nOVERALL SUMMARY:")
display(overall_summary)

print("\nCONDITION SUMMARY:")
display(condition_summary)

print("\nPAIRED CHEXPERT-F1:")
display(stats_df)

print("\nSaved:")
print(overall_path)
print(condition_path)
print(paired_path)
print(stats_path)

print("\nCHEXPERT PHASE26D FINAL AUDIT: PASS")

Input records: 960
Reference vector audit: PASS

Lambda counts:
{0.0: 480, 0.25: 480}

Condition counts:
condition
conflicting               160
evidentiary_incomplete    160
insufficient              160
irrelevant                160
sufficient                160
syntactic_incomplete      160
Name: count, dtype: int64

CHEXPERT RECORD AUDIT: PASS

RECORD RESULTS:
/content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_chexpert.csv

OVERALL SUMMARY:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,lambda,n,f1_mean,f1_median
0,0.00,480,0.247887,0.0
1,0.25,480,0.243165,0.0



CONDITION SUMMARY:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,condition,lambda,n,f1_mean,f1_median
0,conflicting,0.00,80,0.207083,0.0
1,conflicting,0.25,80,0.195000,0.0
2,evidentiary_incomplete,0.00,80,0.308036,0.0
3,evidentiary_incomplete,0.25,80,0.255238,0.0
4,insufficient,0.00,80,0.273452,0.0
5,insufficient,0.25,80,0.273452,0.0
6,irrelevant,0.00,80,0.201786,0.0
7,irrelevant,0.25,80,0.224881,0.0
8,sufficient,0.00,80,0.258393,0.0
9,sufficient,0.25,80,0.257262,0.0



PAIRED CHEXPERT-F1:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,metric,n_pairs,n_nonzero_differences,lambda0_mean,lambda025_mean,mean_delta_lambda025_minus_lambda0,median_delta,wilcoxon_statistic,wilcoxon_p_value,rank_biserial
0,chexpert_f1,480,128,0.247887,0.243165,-0.004722,0.0,3973.0,0.711959,-0.092828



Saved:
/content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_chexpert_summary.csv
/content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_chexpert_by_condition.csv
/content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_chexpert_paired.csv
/content/Evidence-State-Radiology/results/tables/phase26d_metrics/phase26d_evaluation_chexpert_paired_stats.csv

CHEXPERT PHASE26D FINAL AUDIT: PASS


In [ ]:
# ============================================================
# Phase26D — Master Evaluation Synthesis
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

MASTER_DIR = (
    ROOT / "results/tables/phase26d_master"
)
MASTER_DIR.mkdir(parents=True, exist_ok=True)

CLAIM_DIR = (
    ROOT / "results/tables/phase26d_claim_evaluation"
)

METRIC_DIR = (
    ROOT / "results/tables/phase26d_metrics"
)

# ------------------------------------------------------------
# 1. Load claim-level evaluation
# ------------------------------------------------------------

claim_summary = pd.read_csv(
    CLAIM_DIR / "phase26d_evaluation_claim_summary.csv"
)

print("Claim summary:")
display(claim_summary)

# ------------------------------------------------------------
# 2. Load RadGraph summary
# ------------------------------------------------------------

radgraph_summary = pd.read_csv(
    METRIC_DIR
    / "phase26d_evaluation_radgraph_summary.csv"
)

radgraph_paired = pd.read_csv(
    METRIC_DIR
    / "phase26d_evaluation_radgraph_paired_stats.csv"
)

print("\nRadGraph summary:")
display(radgraph_summary)

print("\nRadGraph paired statistics:")
display(radgraph_paired)

# ------------------------------------------------------------
# 3. Load CheXpert summary
# ------------------------------------------------------------

chexpert_summary = pd.read_csv(
    METRIC_DIR
    / "phase26d_evaluation_chexpert_summary.csv"
)

chexpert_paired = pd.read_csv(
    METRIC_DIR
    / "phase26d_evaluation_chexpert_paired_stats.csv"
)

print("\nCheXpert summary:")
display(chexpert_summary)

print("\nCheXpert paired statistics:")
display(chexpert_paired)

# ------------------------------------------------------------
# 4. Extract lambda rows
# ------------------------------------------------------------

claim0 = claim_summary[
    claim_summary["lambda"] == 0.00
].iloc[0]

claim025 = claim_summary[
    claim_summary["lambda"] == 0.25
].iloc[0]

rg0 = radgraph_summary[
    radgraph_summary["lambda"] == 0.00
].iloc[0]

rg025 = radgraph_summary[
    radgraph_summary["lambda"] == 0.25
].iloc[0]

cx0 = chexpert_summary[
    chexpert_summary["lambda"] == 0.00
].iloc[0]

cx025 = chexpert_summary[
    chexpert_summary["lambda"] == 0.25
].iloc[0]

# ------------------------------------------------------------
# 5. Construct master comparison table
# ------------------------------------------------------------

master = pd.DataFrame(
    [
        {
            "metric": "Unsupported claim rate",
            "lambda_0": claim0["unsupported_claim_rate"],
            "lambda_025": claim025["unsupported_claim_rate"],
            "absolute_delta":
                claim025["unsupported_claim_rate"]
                - claim0["unsupported_claim_rate"],
            "relative_change":
                (
                    claim025["unsupported_claim_rate"]
                    - claim0["unsupported_claim_rate"]
                )
                / claim0["unsupported_claim_rate"],
            "paired_test": "Wilcoxon",
            "p_value": 0.318635,
            "interpretation":
                "Slight reduction; not statistically significant",
        },
        {
            "metric": "RadGraph precision",
            "lambda_0": rg0["precision_mean"],
            "lambda_025": rg025["precision_mean"],
            "absolute_delta":
                rg025["precision_mean"]
                - rg0["precision_mean"],
            "relative_change":
                (
                    rg025["precision_mean"]
                    - rg0["precision_mean"]
                )
                / rg0["precision_mean"],
            "paired_test": "Not separately tested",
            "p_value": np.nan,
            "interpretation":
                "Small decrease",
        },
        {
            "metric": "RadGraph recall",
            "lambda_0": rg0["recall_mean"],
            "lambda_025": rg025["recall_mean"],
            "absolute_delta":
                rg025["recall_mean"]
                - rg0["recall_mean"],
            "relative_change":
                (
                    rg025["recall_mean"]
                    - rg0["recall_mean"]
                )
                / rg0["recall_mean"],
            "paired_test": "Not separately tested",
            "p_value": np.nan,
            "interpretation":
                "Small decrease",
        },
        {
            "metric": "RadGraph F1",
            "lambda_0": rg0["f1_mean"],
            "lambda_025": rg025["f1_mean"],
            "absolute_delta":
                rg025["f1_mean"]
                - rg0["f1_mean"],
            "relative_change":
                (
                    rg025["f1_mean"]
                    - rg0["f1_mean"]
                )
                / rg0["f1_mean"],
            "paired_test": "Wilcoxon signed-rank",
            "p_value":
                radgraph_paired.iloc[0]["wilcoxon_p_value"],
            "interpretation":
                "Small increase; not statistically significant",
        },
        {
            "metric": "Custom CheXpert-style F1",
            "lambda_0": cx0["f1_mean"],
            "lambda_025": cx025["f1_mean"],
            "absolute_delta":
                cx025["f1_mean"]
                - cx0["f1_mean"],
            "relative_change":
                (
                    cx025["f1_mean"]
                    - cx0["f1_mean"]
                )
                / cx0["f1_mean"],
            "paired_test": "Wilcoxon signed-rank",
            "p_value":
                chexpert_paired.iloc[0]["wilcoxon_p_value"],
            "interpretation":
                "Small decrease; not statistically significant",
        },
    ]
)

# ------------------------------------------------------------
# 6. Save master table
# ------------------------------------------------------------

master_path = (
    MASTER_DIR
    / "phase26d_master_evaluation.csv"
)

master.to_csv(
    master_path,
    index=False,
)

# ------------------------------------------------------------
# 7. Create concise paper-ready table
# ------------------------------------------------------------

paper_table = master[
    [
        "metric",
        "lambda_0",
        "lambda_025",
        "absolute_delta",
        "p_value",
        "interpretation",
    ]
].copy()

paper_path = (
    MASTER_DIR
    / "phase26d_paper_results_table.csv"
)

paper_table.to_csv(
    paper_path,
    index=False,
)

# ------------------------------------------------------------
# 8. Create condition-level combined table
# ------------------------------------------------------------

claim_condition = pd.read_csv(
    CLAIM_DIR
    / "phase26d_evaluation_claim_summary_by_condition.csv"
)

rg_condition = pd.read_csv(
    METRIC_DIR
    / "phase26d_evaluation_radgraph_by_condition.csv"
)

cx_condition = pd.read_csv(
    METRIC_DIR
    / "phase26d_evaluation_chexpert_by_condition.csv"
)

# Keep only lambda 0 and .25 and merge all metrics.
condition_master = (
    claim_condition[
        [
            "condition",
            "lambda",
            "unsupported_claim_count",
            "unsupported_claim_rate",
        ]
    ]
    .merge(
        rg_condition[
            [
                "condition",
                "lambda",
                "f1_mean",
                "f1_median",
            ]
        ].rename(
            columns={
                "f1_mean": "radgraph_f1_mean",
                "f1_median": "radgraph_f1_median",
            }
        ),
        on=["condition", "lambda"],
        how="inner",
    )
    .merge(
        cx_condition[
            [
                "condition",
                "lambda",
                "f1_mean",
                "f1_median",
            ]
        ].rename(
            columns={
                "f1_mean": "chexpert_f1_mean",
                "f1_median": "chexpert_f1_median",
            }
        ),
        on=["condition", "lambda"],
        how="inner",
    )
    .sort_values(["condition", "lambda"])
    .reset_index(drop=True)
)

condition_path = (
    MASTER_DIR
    / "phase26d_master_by_condition.csv"
)

condition_master.to_csv(
    condition_path,
    index=False,
)

# ------------------------------------------------------------
# 9. Final structural checks
# ------------------------------------------------------------

assert len(master) == 5
assert len(paper_table) == 5
assert len(condition_master) == 12

assert set(condition_master["lambda"]) == {
    0.00,
    0.25,
}

print("\nMASTER TABLE:")
display(master)

print("\nPAPER-READY TABLE:")
display(paper_table)

print("\nCONDITION MASTER:")
display(condition_master)

print("\nSaved:")
print(master_path)
print(paper_path)
print(condition_path)

print("\nPHASE26D MASTER SYNTHESIS AUDIT: PASS")

Claim summary:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,lambda,comparison_rows,supported,omission,extra_negation,ungrounded_affirmation,unsupported_affirmation,contradiction,unclear,unsupported_claim_count,unsupported_claim_rate_denominator,unsupported_claim_rate
0,0.00,1940,471,1005,269,91,35,45,24,171,935,0.182888
1,0.25,1924,479,993,271,77,38,53,13,168,931,0.180451



RadGraph summary:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

,metric,lambda_0,lambda_025,absolute_delta,p_value,interpretation
0,Unsupported claim rate,0.182888,0.180451,-0.002437,0.318635,Slight reduction; not statistically significant
1,RadGraph precision,0.188116,0.185867,-0.002250,NaN,Small decrease
2,RadGraph recall,0.165754,0.164830,-0.000924,NaN,Small decrease
3,RadGraph F1,0.118235,0.120467,0.002232,0.336329,Small increase; not statistically significant
4,Custom CheXpert-style F1,0.247887,0.243165,-0.004722,0.711959,Small decrease; not statistically significant



CONDITION MASTER:


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,condition,lambda,unsupported_claim_count,unsupported_claim_rate,radgraph_f1_mean,radgraph_f1_median,chexpert_f1_mean,chexpert_f1_median
0,conflicting,0.00,46,0.335766,0.097833,0.076923,0.207083,0.0
1,conflicting,0.25,50,0.333333,0.097971,0.084220,0.195000,0.0
2,evidentiary_incomplete,0.00,33,0.160976,0.158154,0.142857,0.308036,0.0
3,evidentiary_incomplete,0.25,29,0.168605,0.156858,0.155870,0.255238,0.0
4,insufficient,0.00,16,0.333333,0.030178,0.000000,0.273452,0.0
5,insufficient,0.25,16,0.333333,0.030178,0.000000,0.273452,0.0
6,irrelevant,0.00,25,0.135870,0.156038,0.130496,0.201786,0.0
7,irrelevant,0.25,23,0.116162,0.142981,0.151923,0.224881,0.0
8,sufficient,0.00,31,0.162304,0.147927,0.142857,0.258393,0.0
9,sufficient,0.25,23,0.119792,0.160245,0.148171,0.257262,0.0



Saved:
/content/Evidence-State-Radiology/results/tables/phase26d_master/phase26d_master_evaluation.csv
/content/Evidence-State-Radiology/results/tables/phase26d_master/phase26d_paper_results_table.csv
/content/Evidence-State-Radiology/results/tables/phase26d_master/phase26d_master_by_condition.csv

PHASE26D MASTER SYNTHESIS AUDIT: PASS


In [ ]:
%cd /content/Evidence-State-Radiology
!git status --short
!git branch --show-current
!git log -1 --oneline

/content/Evidence-State-Radiology
?? docs/phase26d_lambda_selection_policy.md
?? results/maira2_contrastive/
?? results/tables/phase26d/
?? results/tables/phase26d_claim_evaluation/
?? results/tables/phase26d_master/
?? results/tables/phase26d_metrics/
phase25b-maira2-feasibility
5907f33 (HEAD -> phase25b-maira2-feasibility, origin/phase25b-maira2-feasibility) feat: add phases 17-25 reliability evaluation and MAIRA-2 feasibility


In [ ]:
!git status --short

?? docs/phase26d_lambda_selection_policy.md
?? results/maira2_contrastive/
?? results/tables/phase26d/
?? results/tables/phase26d_claim_evaluation/
?? results/tables/phase26d_master/
?? results/tables/phase26d_metrics/


In [ ]:
%cd /content/Evidence-State-Radiology

!echo "===== PHASE 26D FILES ====="
!find docs/phase26d_lambda_selection_policy.md \
       results/maira2_contrastive \
       results/tables/phase26d \
       results/tables/phase26d_claim_evaluation \
       results/tables/phase26d_master \
       results/tables/phase26d_metrics \
       -type f -printf "%p\t%k KB\n" | sort

!echo ""
!echo "===== LARGE FILES (>50 MB) ====="
!find docs/phase26d_lambda_selection_policy.md \
       results/maira2_contrastive \
       results/tables/phase26d \
       results/tables/phase26d_claim_evaluation \
       results/tables/phase26d_master \
       results/tables/phase26d_metrics \
       -type f -size +50M -printf "%p\t%k KB\n" | sort

/content/Evidence-State-Radiology
===== PHASE 26D FILES =====
docs/phase26d_lambda_selection_policy.md	4 KB
results/maira2_contrastive/phase26d_evaluation/phase26d_evaluation_generation.csv	624 KB
results/tables/phase26d_claim_evaluation/phase26d_evaluation_claims.csv	284 KB
results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary_by_condition.csv	4 KB
results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary.csv	4 KB
results/tables/phase26d_claim_evaluation/phase26d_evaluation_paired_records.csv	32 KB
results/tables/phase26d_claim_evaluation/phase26d_lambda_selection.csv	4 KB
results/tables/phase26d_master/phase26d_master_by_condition.csv	4 KB
results/tables/phase26d_master/phase26d_master_evaluation.csv	4 KB
results/tables/phase26d_master/phase26d_paper_results_table.csv	4 KB
results/tables/phase26d_metrics/phase26d_evaluation_chexpert_by_condition.csv	4 KB
results/tables/phase26d_metrics/phase26d_evaluation_chexpert.csv	36 KB
results/tables/phase26

In [ ]:
%cd /content/Evidence-State-Radiology

!git add docs/phase26d_lambda_selection_policy.md \
  results/maira2_contrastive/phase26d_evaluation/phase26d_evaluation_generation.csv \
  results/tables/phase26d/phase26d_maira2_split.csv \
  results/tables/phase26d_claim_evaluation/ \
  results/tables/phase26d_master/ \
  results/tables/phase26d_metrics/

!echo "===== STAGED FILES ====="
!git diff --cached --name-status

/content/Evidence-State-Radiology
===== STAGED FILES =====
A	docs/phase26d_lambda_selection_policy.md
A	results/maira2_contrastive/phase26d_evaluation/phase26d_evaluation_generation.csv
A	results/tables/phase26d/phase26d_maira2_split.csv
A	results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary.csv
A	results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary_by_condition.csv
A	results/tables/phase26d_claim_evaluation/phase26d_evaluation_claims.csv
A	results/tables/phase26d_claim_evaluation/phase26d_evaluation_paired_records.csv
A	results/tables/phase26d_claim_evaluation/phase26d_lambda_selection.csv
A	results/tables/phase26d_master/phase26d_master_by_condition.csv
A	results/tables/phase26d_master/phase26d_master_evaluation.csv
A	results/tables/phase26d_master/phase26d_paper_results_table.csv
A	results/tables/phase26d_metrics/phase26d_evaluation_chexpert.csv
A	results/tables/phase26d_metrics/phase26d_evaluation_chexpert_by_condition.csv
A	results/table

In [ ]:
%cd /content/Evidence-State-Radiology

!echo "===== STATUS ====="
!git status --short

!echo ""
!echo "===== STAGED FILE COUNT ====="
!git diff --cached --name-only | wc -l

!echo ""
!echo "===== STAGED SIZE ====="
!git diff --cached --stat

/content/Evidence-State-Radiology
===== STATUS =====
A  docs/phase26d_lambda_selection_policy.md
A  results/maira2_contrastive/phase26d_evaluation/phase26d_evaluation_generation.csv
A  results/tables/phase26d/phase26d_maira2_split.csv
A  results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary.csv
A  results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary_by_condition.csv
A  results/tables/phase26d_claim_evaluation/phase26d_evaluation_claims.csv
A  results/tables/phase26d_claim_evaluation/phase26d_evaluation_paired_records.csv
A  results/tables/phase26d_claim_evaluation/phase26d_lambda_selection.csv
A  results/tables/phase26d_master/phase26d_master_by_condition.csv
A  results/tables/phase26d_master/phase26d_master_evaluation.csv
A  results/tables/phase26d_master/phase26d_paper_results_table.csv
A  results/tables/phase26d_metrics/phase26d_evaluation_chexpert.csv
A  results/tables/phase26d_metrics/phase26d_evaluation_chexpert_by_condition.csv
A  resul

In [ ]:
%cd /content/Evidence-State-Radiology

!git commit -m "results: add Phase 26D MAIRA-2 evaluation artifacts"

/content/Evidence-State-Radiology
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@873988c3d1b3.(none)')


In [ ]:
%cd /content/Evidence-State-Radiology

!git config user.name "Vasu1912"
!git config user.email "gangavarapuvasu.2005@gmail.com"

print("Git identity:")
!git config user.name
!git config user.email

/content/Evidence-State-Radiology
Git identity:
Vasu1912
gangavarapuvasu.2005@gmail.com


In [ ]:
!git commit -m "results: add Phase 26D MAIRA-2 evaluation artifacts"

[phase25b-maira2-feasibility c660f21] results: add Phase 26D MAIRA-2 evaluation artifacts
 21 files changed, 9357 insertions(+)
 create mode 100644 docs/phase26d_lambda_selection_policy.md
 create mode 100644 results/maira2_contrastive/phase26d_evaluation/phase26d_evaluation_generation.csv
 create mode 100644 results/tables/phase26d/phase26d_maira2_split.csv
 create mode 100644 results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary.csv
 create mode 100644 results/tables/phase26d_claim_evaluation/phase26d_evaluation_claim_summary_by_condition.csv
 create mode 100644 results/tables/phase26d_claim_evaluation/phase26d_evaluation_claims.csv
 create mode 100644 results/tables/phase26d_claim_evaluation/phase26d_evaluation_paired_records.csv
 create mode 100644 results/tables/phase26d_claim_evaluation/phase26d_lambda_selection.csv
 create mode 100644 results/tables/phase26d_master/phase26d_master_by_condition.csv
 create mode 100644 results/tables/phase26d_master/phase26d_m

In [ ]:
!git status --short
!git log -2 --oneline

c660f21 (HEAD -> phase25b-maira2-feasibility) results: add Phase 26D MAIRA-2 evaluation artifacts
5907f33 (origin/phase25b-maira2-feasibility) feat: add phases 17-25 reliability evaluation and MAIRA-2 feasibility


In [ ]:
%cd /content/Evidence-State-Radiology

!git push origin phase25b-maira2-feasibility

/content/Evidence-State-Radiology
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
%cd /content/Evidence-State-Radiology

!git bundle create /content/phase26d_checkpoint.bundle 5907f33..c660f21

!ls -lh /content/phase26d_checkpoint.bundle

/content/Evidence-State-Radiology
fatal: Refusing to create empty bundle.
ls: cannot access '/content/phase26d_checkpoint.bundle': No such file or directory


In [ ]:
%cd /content/Evidence-State-Radiology

!git show --no-patch --format=fuller c660f21
!git rev-parse c660f21^

/content/Evidence-State-Radiology
commit c660f21aba26f24c56e5a791fc9ce873db67e537 (HEAD -> phase25b-maira2-feasibility)
Author:     Vasu1912 <gangavarapuvasu.2005@gmail.com>
AuthorDate: Tue Oct 6 16:24:33 2026 +0000
Commit:     Vasu1912 <gangavarapuvasu.2005@gmail.com>
CommitDate: Tue Oct 6 16:24:33 2026 +0000

    results: add Phase 26D MAIRA-2 evaluation artifacts
5907f338ac290a64bbd01c0610a3f8d2b8f13b37


In [ ]:
%cd /content/Evidence-State-Radiology

!git bundle create /content/phase26d_checkpoint.bundle c660f21aba26f24c56e5a791fc9ce873db67e537
!ls -lh /content/phase26d_checkpoint.bundle

/content/Evidence-State-Radiology
fatal: Refusing to create empty bundle.
ls: cannot access '/content/phase26d_checkpoint.bundle': No such file or directory


In [ ]:
%cd /content/Evidence-State-Radiology

!git format-patch --stdout 5907f338ac290a64bbd01c0610a3f8d2b8f13b37..c660f21aba26f24c56e5a791fc9ce873db67e537 > /content/phase26d_commit.patch

!ls -lh /content/phase26d_commit.patch

/content/Evidence-State-Radiology
-rw-r--r-- 1 root root 1.6M Oct  6 16:32 /content/phase26d_commit.patch


In [ ]:
from google.colab import files
files.download("/content/phase26d_commit.patch")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>